# fase_3 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian CRM, Prospek, dan Operasional.

In [16]:
import sys
import os
import mysql.connector
import pandas as pd
import datetime
import random
import string
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## Connect ke Database

In [17]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration
Connected to future database: 2


## Ambil Data dari DB Lama

In [18]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()
target_tables = [list(t.values())[0] for t in tables_data]
print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")

df_old = {}
for table in target_tables:
    try:
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")


--- Ditemukan 108 tabel di Database Lama ---
Berhasil load tabel: absensi | Jumlah baris: 13444
Berhasil load tabel: absensi_note | Jumlah baris: 11
Berhasil load tabel: bidang | Jumlah baris: 4
Berhasil load tabel: bidangkategori | Jumlah baris: 12
Berhasil load tabel: bidanglink | Jumlah baris: 7
Berhasil load tabel: calon | Jumlah baris: 4
Berhasil load tabel: calon_detil | Jumlah baris: 61
Berhasil load tabel: calon_pertanyaan | Jumlah baris: 229
Berhasil load tabel: calon_pertanyaan_detil | Jumlah baris: 4305
Berhasil load tabel: catatan_kelas | Jumlah baris: 12797
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 999
Berhasil load tabel: catatan_mingguan | Jumlah baris: 0
Berhasil load tabel: catatan_siswa | Jumlah baris: 1502
Berhasil load tabel: catatan_siswa_follow_up | Jumlah baris: 22
Berhasil load tabel: catatanawal_admin | Jumlah baris: 64
Berhasil load tabel: catatanawal_datautama | Jumlah baris: 9
Berhasil load tabel: catatanawal_infolain | Jumlah baris: 64
Berhasi

## Ambil Data dari DB Baru (Struktur Target)

In [19]:
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()
target_tables_new = [list(t.values())[0] for t in tables_data_new]
df_new = {}

for table in target_tables_new:
    try:
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 12
Berhasil load tabel: bidang_link | Jumlah baris: 7
Berhasil load tabel: busdev_bidang | Jumlah baris: 4
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 165
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 338
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 338
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 338
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 338
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 338
Berhasil load tabel: calon_siswa_proses | Jumlah baris: 338
Berhasil load tabel: calon_siswa_status_logs | Jumlah baris: 0
Berhasil load tabel: catatan_kelas | Jumlah baris: 25594
Berhasil load tabel: catatan_kelas_tag | Jumlah baris: 0
Ber

In [20]:
cursor_future.execute("SHOW TABLES")
tables_data_future = cursor_future.fetchall()
target_tables_future = [list(t.values())[0] for t in tables_data_future]
df_future = {}

for table in target_tables_future:
    try:
        query = f"SELECT * FROM `{table}`"
        df_future[table] = pd.read_sql(query, db_future)
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_future[table])}")
    except:
        pass

Berhasil load tabel: absensi | Jumlah baris: 0
Berhasil load tabel: activity_log | Jumlah baris: 0
Berhasil load tabel: admin_sarpras | Jumlah baris: 1
Berhasil load tabel: bidang_kategori | Jumlah baris: 0
Berhasil load tabel: bidang_link | Jumlah baris: 0
Berhasil load tabel: busdev_bidang | Jumlah baris: 0
Berhasil load tabel: cache | Jumlah baris: 0
Berhasil load tabel: cache_locks | Jumlah baris: 0
Berhasil load tabel: calon_siswa | Jumlah baris: 0
Berhasil load tabel: calon_siswa_akademik | Jumlah baris: 0
Berhasil load tabel: calon_siswa_bayar | Jumlah baris: 0
Berhasil load tabel: calon_siswa_fo_detail | Jumlah baris: 0
Berhasil load tabel: calon_siswa_form_program_requirements | Jumlah baris: 0
Berhasil load tabel: calon_siswa_form_programs | Jumlah baris: 0
Berhasil load tabel: calon_siswa_jadwal | Jumlah baris: 0
Berhasil load tabel: calon_siswa_kursus | Jumlah baris: 0
Berhasil load tabel: calon_siswa_ortu | Jumlah baris: 0
Berhasil load tabel: calon_siswa_proses | Jumlah b

## Tahap 1: Penggabungan

In [21]:
import pandas as pd

print("=================================================================")
print(" 🚀 FASE 3 REBORN - TAHAP 1: PERSIAPAN TABEL SUMBER MENTAH 🚀 ")
print("=================================================================")

# ===============================================================================
# 1. MEMBUAT TABLE 1: ALL_FORM_DETIL (Gabungan Kertas Formulir Pendaftaran)
# ===============================================================================
def build_all_form_detil(df_old_dict):
    # Base table adalah form_calon
    if 'form_calon' not in df_old_dict:
        print("⚠️ Tabel 'form_calon' tidak ditemukan!")
        return pd.DataFrame()
        
    df_form = df_old_dict['form_calon'].copy()
    join_key = 'idcalon'
    
    # Left join dengan detil 1 sampai 4 secara berurutan
    tabel_detil = ['form_calon_detil1', 'form_calon_detil2', 'form_calon_detil3', 'form_calon_detil4']
    
    for nama_tabel in tabel_detil:
        if nama_tabel in df_old_dict:
            df_form = pd.merge(
                df_form, 
                df_old_dict[nama_tabel], 
                on=join_key, 
                how='left', 
                suffixes=('', f'_{nama_tabel[-4:]}') # Menambahkan suffix jika ada nama kolom kembar
            )
            
    return df_form

# ===============================================================================
# 2. MEMBUAT TABLE 2: ALL_CATATAN_AWAL_ADMIN (Gabungan Buku Tamu / Notes FO)
# ===============================================================================
def build_all_catatan_awal_admin(df_old_dict):
    # Base table adalah catatanawal_admin
    if 'catatanawal_admin' not in df_old_dict:
        print("⚠️ Tabel 'catatanawal_admin' tidak ditemukan!")
        return pd.DataFrame()
        
    df_catatan = df_old_dict['catatanawal_admin'].copy()
    join_key = 'idcatatanawal_admin'
    
    # Left join dengan datautama, infolain, dan tglpenting
    tabel_detil = ['catatanawal_datautama', 'catatanawal_infolain', 'catatanawal_tglpenting']
    
    for nama_tabel in tabel_detil:
        if nama_tabel in df_old_dict:
            df_catatan = pd.merge(
                df_catatan, 
                df_old_dict[nama_tabel], 
                on=join_key, 
                how='left', 
                suffixes=('', f'_{nama_tabel.split("_")[-1]}') # Menambahkan suffix jika ada nama kolom kembar
            )
            
    return df_catatan

# === EKSEKUSI FUNGSI ===
# (Asumsinya data mentah lama kamu ada di dalam dictionary 'df_old')
all_form_detil = build_all_form_detil(df_old)
all_catatan_awal_admin = build_all_catatan_awal_admin(df_old)

# Menampilkan hasil audit 
print(f"✅ TABLE 1 (all_form_detil) berhasil dibuat! Total: {len(all_form_detil)} baris, {len(all_form_detil.columns)} kolom.")
print(f"✅ TABLE 2 (all_catatan_awal_admin) berhasil dibuat! Total: {len(all_catatan_awal_admin)} baris, {len(all_catatan_awal_admin.columns)} kolom.")

print("\n--- Pratinjau Table 1 (all_form_detil) ---")
display(all_form_detil)
display(all_form_detil.info())

print("\n--- Pratinjau Table 2 (all_catatan_awal_admin) ---")
display(all_catatan_awal_admin)
display(all_catatan_awal_admin.info())

 🚀 FASE 3 REBORN - TAHAP 1: PERSIAPAN TABEL SUMBER MENTAH 🚀 
✅ TABLE 1 (all_form_detil) berhasil dibuat! Total: 184 baris, 66 kolom.
✅ TABLE 2 (all_catatan_awal_admin) berhasil dibuat! Total: 64 baris, 54 kolom.

--- Pratinjau Table 1 (all_form_detil) ---


,idcalon,fullName,email,nickName,phone1,phone2,schoolName,classLevel,gender,activities,...,nomor_invoice,bank,tanggal_pembayaran,bulan_masuk,idcalon_detil4,followUp1,followUp2,followUp3,keterangan_til4,akun_lv
0,C00000017,Daria Azmiya Jasmine,puterihapsari.f@gmail.com,Daria,082231346758,,TK Al Maghfirah,TK B,Perempuan,None,...,,Mandiri,2025-09-17,September,Z00000017,None,None,None,,
1,C00000018,Annisa Zahro Ramadhania,dwirohm4@gmail.com,Annisa,081336647476,,Sd Khadijah wonorejo,Kelas 3,Perempuan,None,...,,Mandiri,2025-09-22,September,Z00000018,None,None,None,,
2,C00000019,Zulfa Bariatur Rahma,chyzryth@gmail.com,Zulfa,085707179656,083849241708,SMAN 17 Surabaya,X,None,None,...,,None,None,None,Z00000024,None,None,None,,
3,C00000020,Achmad Naufal Albiruni,melisnifuku@gmail.com,AlBI,087765283592,None,SD Khadijah Wonorejo Surabaya,SD kelas 5,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C00000021,Khansa Amalia Putri Aji,afadhilpa@gmail.com,Khansa,081554932188,,SMAN 17 Surabaya,XI,Perempuan,None,...,,Mandiri,2025-09-29,September,Z00000019,None,None,None,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,C00000197,Antonius Miguel Kurniawan,adeodatus.kurniawan@gmail.com,Miguel,08170326911,None,SD Kartika Nasional Plus,SD Kelas 2,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180,C00000198,Tisha kayla janitra,nroskalindha18@gmail.com,Tisha,082244441630,None,Tk al fajar,Tk b,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,C00000199,Clariza Arifianti,clarizarisa3@gmail.com,Clara,08977257033,None,None,None,Perempuan,Mahasiswa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
182,C00000200,Shaqila Anindra Dzakira,shaqilaanindra@gmail.com,Shaqila,085850209079,None,SDIT Ghilmani Surabaya,kelas 4,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 66 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   idcalon             184 non-null    object         
 1   fullName            184 non-null    object         
 2   email               184 non-null    object         
 3   nickName            184 non-null    object         
 4   phone1              184 non-null    object         
 5   phone2              84 non-null     object         
 6   schoolName          117 non-null    object         
 7   classLevel          117 non-null    object         
 8   gender              176 non-null    object         
 9   activities          67 non-null     object         
 10  otherActivities     67 non-null     object         
 11  curriculum          109 non-null    object         
 12  exp                 117 non-null    object         
 13  diagnostic          109 non-null   

None


--- Pratinjau Table 2 (all_catatan_awal_admin) ---


,idcatatanawal_admin,nama,tlp,email,status,created_at,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,bank,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date
0,M000000001,Ahmad 1,0812312344531,ahmad@gmail.com,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000002,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,done,2025-09-15 13:33:49,2025-09-18 17:39:40,N000000001,Ibu Sari,Daria Azmiya Jasmine,...,Transfer Bank,,,O000000009,2025-09-15,None,2025-09-15,2025-09-17,2025-09-24,None
2,M000000003,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,done,2025-09-16 11:13:41,2025-09-23 16:25:14,N000000002,Bu Dwi,Annisa Zahro Ramadhania,...,,,,O000000014,2025-09-16,2025-09-16,2025-09-16,None,2025-09-25,None
3,M000000004,Ghayda Syakira Hanania,081235160064,humaidah0208@gmail.com,done,2025-09-17 16:35:22,2025-10-03 09:47:53,N000000003,,Ghayda Syakira Hanania,...,,,,O000000012,2025-09-15,2025-09-15,2025-09-15,None,2025-09-25,None
4,M000000005,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,done,2025-09-17 16:49:20,2025-10-03 09:49:15,N000000004,Bu Lita,Achmad Naufal Albiruni,...,,,,O000000013,2025-09-17,2025-09-17,2025-09-17,None,2025-09-25,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,M000000060,Cheryl Kathleen Tan,0811537168,cheryl2013sby@gmail.com,on progress,2025-10-24 15:23:53,2025-10-24 15:23:53,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,M000000061,Bimantara Sukma Arsa,08113322866,asihasih180808@gmail.com,on progress,2025-10-24 15:24:48,2025-10-24 15:24:48,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,M000000062,Aca,082264644495,,follow up another time,2025-10-24 15:27:54,2025-10-24 15:27:54,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,M000000063,Bu Hartik,0895368241226,,waiting for confirmation,2025-10-24 15:28:21,2025-10-24 15:28:21,NaN,NaN,NaN,...,,,,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 54 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   idcatatanawal_admin       64 non-null     object        
 1   nama                      64 non-null     object        
 2   tlp                       64 non-null     object        
 3   email                     64 non-null     object        
 4   status                    64 non-null     object        
 5   created_at                64 non-null     datetime64[ns]
 6   updated_at                64 non-null     datetime64[ns]
 7   idcatatanawal_datautama   9 non-null      object        
 8   pengontak_admin           9 non-null      object        
 9   nama_l                    9 non-null      object        
 10  nama_p                    9 non-null      object        
 11  jenis_k                   9 non-null      object        
 12  no_wa_ortu              

None

In [22]:
print("=================================================================")
print(" 🚀 FASE 3 REBORN - TAHAP 2: PERKAWINAN DATA (TABLE 3 & 4) 🚀 ")
print("=================================================================")

# ===============================================================================
# A. PEMBERSIHAN KUNCI (NATURAL KEY) SEBELUM DI-JOIN
# ===============================================================================
# Kita gunakan EMAIL sebagai kunci utama perkawinan.
# Kita ubah jadi huruf kecil semua (.lower()) dan hilangkan spasi nyasar (.strip())

all_form_detil['join_email'] = all_form_detil['email'].astype(str).str.lower().str.strip()
all_catatan_awal_admin['join_email'] = all_catatan_awal_admin['email'].astype(str).str.lower().str.strip()

# ===============================================================================
# B. MEMBUAT TABLE 3: LEFT JOIN (Anak yang ngisi Form + Histori FO)
# ===============================================================================
# Base table: all_form_detil
table_3_left_join = pd.merge(
    all_form_detil, 
    all_catatan_awal_admin, 
    on='join_email', 
    how='left', 
    suffixes=('_form', '_admin') # Jika ada kolom sama, tambahkan penanda
)

# ===============================================================================
# C. MEMBUAT TABLE 4: LEFT EXCLUSIVE JOIN (Anak yang HANYA tanya ke FO)
# ===============================================================================
# Base table: all_catatan_awal_admin
temp_exclusive_merge = pd.merge(
    all_catatan_awal_admin, 
    all_form_detil, 
    on='join_email', 
    how='left', 
    indicator=True, 
    suffixes=('_admin', '_form')
)

# Ambil HANYA yang berasal dari 'left_only' (Catatan FO saja, tidak match ke Form)
table_4_exclusive = temp_exclusive_merge[temp_exclusive_merge['_merge'] == 'left_only'].copy()

# 🎯 PERBAIKAN CIMUT: Reset nomor index agar urutannya kembali rapi (0, 1, 2, 3... 27)
table_4_exclusive = table_4_exclusive.reset_index(drop=True)

# Buang kolom '_merge' karena sudah tidak dipakai
table_4_exclusive = table_4_exclusive.drop(columns=['_merge'])

# ===============================================================================
# D. AUDIT HASIL PERKAWINAN
# ===============================================================================
print(f"✅ TABLE 3 (Form + FO) dibuat! Total: {len(table_3_left_join)} baris.")
print(f"✅ TABLE 4 (Hanya FO / Ghosting) dibuat! Total: {len(table_4_exclusive)} baris.")

# Cek berapa banyak anak di Table 3 yang berhasil ditarik data FO-nya
jumlah_match_fo = table_3_left_join['idcatatanawal_admin'].notna().sum()
print(f"   -> Dari {len(table_3_left_join)} anak di form, ada {jumlah_match_fo} anak yang rekam jejak FO-nya berhasil dikawinkan!")

print("\n--- Pratinjau Table 3 ---")
display(table_3_left_join)
display(table_3_left_join.info(verbose=True, show_counts=True))

print("\n--- Pratinjau Table 4 ---")
display(table_4_exclusive)
display(table_4_exclusive.info(verbose=True, show_counts=True))

 🚀 FASE 3 REBORN - TAHAP 2: PERKAWINAN DATA (TABLE 3 & 4) 🚀 
✅ TABLE 3 (Form + FO) dibuat! Total: 186 baris.
✅ TABLE 4 (Hanya FO / Ghosting) dibuat! Total: 28 baris.
   -> Dari 186 anak di form, ada 43 anak yang rekam jejak FO-nya berhasil dikawinkan!

--- Pratinjau Table 3 ---


,idcalon,fullName,email_form,nickName,phone1,phone2,schoolName,classLevel,gender,activities,...,bank_admin,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,trial_date,tglbayar_date,tglmasuk_date,tglkeluar_date
0,C00000017,Daria Azmiya Jasmine,puterihapsari.f@gmail.com,Daria,082231346758,,TK Al Maghfirah,TK B,Perempuan,None,...,Transfer Bank,,,O000000009,2025-09-15,None,2025-09-15,2025-09-17,2025-09-24,None
1,C00000018,Annisa Zahro Ramadhania,dwirohm4@gmail.com,Annisa,081336647476,,Sd Khadijah wonorejo,Kelas 3,Perempuan,None,...,,,,O000000014,2025-09-16,2025-09-16,2025-09-16,None,2025-09-25,None
2,C00000019,Zulfa Bariatur Rahma,chyzryth@gmail.com,Zulfa,085707179656,083849241708,SMAN 17 Surabaya,X,None,None,...,,,,O000000007,2025-09-16,None,None,None,None,None
3,C00000020,Achmad Naufal Albiruni,melisnifuku@gmail.com,AlBI,087765283592,None,SD Khadijah Wonorejo Surabaya,SD kelas 5,Laki-laki,None,...,,,,O000000013,2025-09-17,2025-09-17,2025-09-17,None,2025-09-25,None
4,C00000021,Khansa Amalia Putri Aji,afadhilpa@gmail.com,Khansa,081554932188,,SMAN 17 Surabaya,XI,Perempuan,None,...,,,,O000000017,2025-09-15,2025-09-18,2025-09-18,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,C00000197,Antonius Miguel Kurniawan,adeodatus.kurniawan@gmail.com,Miguel,08170326911,None,SD Kartika Nasional Plus,SD Kelas 2,Laki-laki,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
182,C00000198,Tisha kayla janitra,nroskalindha18@gmail.com,Tisha,082244441630,None,Tk al fajar,Tk b,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
183,C00000199,Clariza Arifianti,clarizarisa3@gmail.com,Clara,08977257033,None,None,None,Perempuan,Mahasiswa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
184,C00000200,Shaqila Anindra Dzakira,shaqilaanindra@gmail.com,Shaqila,085850209079,None,SDIT Ghilmani Surabaya,kelas 4,Perempuan,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 121 columns):
 #    Column                    Non-Null Count  Dtype          
---   ------                    --------------  -----          
 0    idcalon                   186 non-null    object         
 1    fullName                  186 non-null    object         
 2    email_form                186 non-null    object         
 3    nickName                  186 non-null    object         
 4    phone1                    186 non-null    object         
 5    phone2                    86 non-null     object         
 6    schoolName                119 non-null    object         
 7    classLevel                119 non-null    object         
 8    gender                    178 non-null    object         
 9    activities                67 non-null     object         
 10   otherActivities           67 non-null     object         
 11   curriculum                111 non-null    object        

None


--- Pratinjau Table 4 ---


,idcatatanawal_admin,nama,tlp,email_admin,status_admin,created_at_admin,updated_at,idcatatanawal_datautama,pengontak_admin,nama_l,...,nomor_invoice,bank_form,tanggal_pembayaran,bulan_masuk_form,idcalon_detil4,followUp1,followUp2,followUp3,keterangan_til4,akun_lv
0,M000000001,Ahmad 1,0812312344531,ahmad@gmail.com,canceled,2025-09-12 00:51:16,2025-09-12 00:51:16,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000000017,Bu Tika,089677819546,,follow up another time,2025-09-24 14:02:51,2025-09-24 14:02:51,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M000000018,Bu Devi,081340370088,,follow up another time,2025-09-24 14:07:30,2025-09-24 14:07:30,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M000000020,Jenah,085786164620,,follow up another time,2025-10-03 11:40:19,2025-10-03 11:40:19,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M000000026,Inna,081313223427,,follow up another time,2025-10-03 12:39:02,2025-10-03 12:39:02,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,M000000027,Feb,081231225959,,follow up another time,2025-10-03 12:41:41,2025-10-03 12:41:41,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,M000000028,Intan,085219083605,,canceled,2025-10-03 15:40:27,2025-10-03 15:42:26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,M000000029,Herni,088217919663,,follow up another time,2025-10-03 15:43:23,2025-10-03 15:43:23,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,M000000030,Sisca,085755891737,,follow up another time,2025-10-03 15:44:18,2025-10-03 15:44:18,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,M000000031,Tilla,085706406904,,follow up another time,2025-10-03 15:45:13,2025-10-03 15:45:13,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 121 columns):
 #    Column                    Non-Null Count  Dtype          
---   ------                    --------------  -----          
 0    idcatatanawal_admin       28 non-null     object         
 1    nama                      28 non-null     object         
 2    tlp                       28 non-null     object         
 3    email_admin               28 non-null     object         
 4    status_admin              28 non-null     object         
 5    created_at_admin          28 non-null     datetime64[ns] 
 6    updated_at                28 non-null     datetime64[ns] 
 7    idcatatanawal_datautama   0 non-null      object         
 8    pengontak_admin           0 non-null      object         
 9    nama_l                    0 non-null      object         
 10   nama_p                    0 non-null      object         
 11   jenis_k                   0 non-null      object         


None

# TAHAP 2 PEMBERSIHAN

In [23]:
import pandas as pd
import numpy as np

print("=================================================================")
print(" 🚀 RADAR PENDETEKSI DUPLIKAT & KASUS KAKAK-ADIK (1 EMAIL) 🚀 ")
print("=================================================================")

# Gunakan copy dari table 3
df_cek = table_3_left_join.copy()

# ===============================================================================
# 1. STANDARISASI TEKS (Sapu Bersih Spasi & Huruf Besar)
# ===============================================================================
def bersihkan_teks(kolom):
    # Ubah jadi string -> huruf kecil -> hapus spasi ujung -> ganti 'nan'/'none' jadi Kosong
    return df_cek[kolom].astype(str).str.lower().str.strip().replace(['nan', 'none', ''], np.nan)

df_cek['cek_fullname'] = bersihkan_teks('fullName')
df_cek['cek_nama_l'] = bersihkan_teks('nama_l')
df_cek['cek_email'] = bersihkan_teks('join_email')

# ===============================================================================
# 2. RADAR DETEKSI ANOMALI
# ===============================================================================

# 🚩 DETEKSI 1: Beda Nama Form vs FO di Email yang Sama (Kasus Kakak-Adik / Typo)
# Syarat: Keduanya punya isi (tidak NaN), tapi teksnya berbeda
df_cek['is_beda_form_fo'] = np.where(
    df_cek['cek_fullname'].notna() & 
    df_cek['cek_nama_l'].notna() & 
    (df_cek['cek_fullname'] != df_cek['cek_nama_l']), 
    True, False
)

# 🚩 DETEKSI 2: Satu Email Dipakai Banyak Nama (Akun Keroyokan / Sibling)
# Menghitung ada berapa 'fullName' unik di dalam 1 email
email_nama_unik = df_cek.groupby('cek_email')['cek_fullname'].transform('nunique')
df_cek['is_email_keroyokan'] = np.where(email_nama_unik > 1, True, False)

# 🚩 DETEKSI 3: Duplikat Murni (Double Submit)
# Nama dan Email sama persis, tapi muncul lebih dari 1 kali
df_cek['is_duplikat_murni'] = df_cek.duplicated(subset=['cek_fullname', 'cek_email'], keep=False)

# ===============================================================================
# 3. FILTER DAN TAMPILKAN HASILNYA
# ===============================================================================

# Ambil baris yang menyala di salah satu radar di atas
data_mencurigakan = df_cek[
    df_cek['is_beda_form_fo'] | 
    df_cek['is_email_keroyokan'] | 
    df_cek['is_duplikat_murni']
].copy()

# Sortir berdasarkan email agar keluarga/duplikat berjejer berdekatan
data_mencurigakan = data_mencurigakan.sort_values(by=['cek_email', 'cek_fullname'])

# Pilih kolom-kolom krusial untuk dipelototi
kolom_investigasi = [
    'idcalon', 'join_email', 
    'fullName', 'nama_l',  # Bandingkan Nama Lengkap
    'nickName', 'nama_p',  # Bandingkan Nama Panggilan
    'phone1', 'no_wa_ortu',# Bandingkan Nomor HP
    'is_beda_form_fo', 'is_email_keroyokan', 'is_duplikat_murni'
]

print(f"🚨 Radar menemukan {len(data_mencurigakan)} baris yang mencurigakan!")
print("Silakan geser ke kanan untuk melihat status radarnya (True/False).")
display(data_mencurigakan[kolom_investigasi])

 🚀 RADAR PENDETEKSI DUPLIKAT & KASUS KAKAK-ADIK (1 EMAIL) 🚀 
🚨 Radar menemukan 51 baris yang mencurigakan!
Silakan geser ke kanan untuk melihat status radarnya (True/False).


,idcalon,join_email,fullName,nama_l,nickName,nama_p,phone1,no_wa_ortu,is_beda_form_fo,is_email_keroyokan,is_duplikat_murni
150,C00000166,anikewulansari@yahoo.co.id,Adyasta Sakti Rajendra Permana,NaN,Dyas,NaN,081217042400,NaN,False,True,False
149,C00000165,anikewulansari@yahoo.co.id,Kikandrya Charmaraiza Permana,NaN,Raiza,NaN,081217042400,NaN,False,True,False
98,C00000114,berri.fauzian13@gmail.com,Evangelina Nailah Putri Fauzian,NaN,Nailah,NaN,081235236889,NaN,False,True,False
99,C00000115,berri.fauzian13@gmail.com,Shafiyah Anindyah Putri Fauzian,NaN,Shafiyah,NaN,081235236886,NaN,False,True,False
2,C00000019,chyzryth@gmail.com,Zulfa Bariatur Rahma,Zulfa Bari'atur Rahma,Zulfa,Zulfa,085707179656,083849241708,True,False,False
44,C00000060,data@gmail.com,tes bibah,NaN,tes bibah,NaN,085730933317,NaN,False,True,True
49,C00000065,data@gmail.com,tes bibah,NaN,tes bibah,NaN,085730933317,NaN,False,True,True
40,C00000056,data@gmail.com,tes bibah hapus,NaN,tes bibah hapus,NaN,0857456556232,NaN,False,True,False
27,C00000043,dianamandrasari@gmail.com,Diana Mandasari,NaN,Diana,NaN,0896-1347-1313,NaN,False,False,True
28,C00000044,dianamandrasari@gmail.com,Diana Mandasari,NaN,Diana,NaN,0896-1347-1313,NaN,False,False,True


In [24]:
import pandas as pd

print("=================================================================")
print(" 🚀 FASE 3 REBORN - TAHAP 2.8: TEBAS DUPLIKAT & MERGE MANUAL 🚀 ")
print("=================================================================")

# 1. DAFTAR ID YANG AKAN DITEBAS (DROP)
ids_to_drop = [
    'C00000060', 'C00000065', 'C00000056', 'C00000044', 'C00000045',
    'C00000186', 'C00000188', 'C00000189', 'C00000190', 'C00000193',
    'C00000057', 'C00000113', 'C00000039', 'C00000125', 'C00000058',
    'C00000063', 'C00000100', 'C00000037'
]

# Hitung jumlah awal
jml_awal = len(table_3_left_join)

# Eksekusi Tebas Baris
table_3_left_join = table_3_left_join[~table_3_left_join['idcalon'].isin(ids_to_drop)].copy()

jml_ditebas = jml_awal - len(table_3_left_join)
print(f"✅ BERHASIL: {jml_ditebas} baris duplikat/kotor telah resmi ditebas dari table_3_left_join!")

# ===============================================================================
# 2. EKSEKUSI MERGE MANUAL (C00000036 & C00000035)
# ===============================================================================
# Kita ambil dulu kedua baris tersebut (jika ada)
row_36 = table_3_left_join[table_3_left_join['idcalon'] == 'C00000036']
row_35 = table_3_left_join[table_3_left_join['idcalon'] == 'C00000035']

if not row_36.empty and not row_35.empty:
    print("\n🔍 Menemukan C00000036 dan C00000035. Memulai proses merge...")
    
    # Jadikan C00000036 sebagai data utama (Series), lalu tambal kekurangannya pakai C00000035
    merged_series = row_36.iloc[0].combine_first(row_35.iloc[0])
    
    # Masukkan hasil gabungan kembali ke posisi C00000036 di tabel utama
    idx_36 = row_36.index[0]
    table_3_left_join.loc[idx_36] = merged_series
    
    # Hapus baris C00000035 karena datanya sudah diserap
    table_3_left_join = table_3_left_join[table_3_left_join['idcalon'] != 'C00000035']
    
    print("✅ BERHASIL: C00000035 telah dilebur ke dalam C00000036 dan dihapus dari tabel!")
else:
    print("\n⚠️ Lewati merge: Salah satu atau kedua ID (C00000036 / C00000035) tidak ditemukan di tabel.")

# Reset index agar rapi kembali
table_3_left_join = table_3_left_join.reset_index(drop=True)

print(f"\n📊 STATUS AKHIR: table_3_left_join sekarang memiliki {len(table_3_left_join)} baris bersih.")

 🚀 FASE 3 REBORN - TAHAP 2.8: TEBAS DUPLIKAT & MERGE MANUAL 🚀 
✅ BERHASIL: 18 baris duplikat/kotor telah resmi ditebas dari table_3_left_join!

🔍 Menemukan C00000036 dan C00000035. Memulai proses merge...
✅ BERHASIL: C00000035 telah dilebur ke dalam C00000036 dan dihapus dari tabel!

📊 STATUS AKHIR: table_3_left_join sekarang memiliki 166 baris bersih.


In [25]:
import pandas as pd
import numpy as np

print("=================================================================")
print(" 🚀 FASE 3 REBORN - TAHAP 2.5: PRE-PROCESSING NOMOR HP 🚀 ")
print("=================================================================")

def prep_nomor_hp(df, nama_tabel):
    df_clean = df.copy()
    
    # Daftar kolom target yang akan diproses
    kolom_hp = ['phone1', 'phone2', 'no_wa_ortu', 'no_wa_anak']
    
    # 1. Bersihkan dulu string kosong, '-', atau 'nan' khusus di kolom HP agar fallback-nya akurat
    for col in kolom_hp:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].replace(r'^\s*$', np.nan, regex=True)
            df_clean[col] = df_clean[col].replace(['-', 'nan', 'None', ''], np.nan)
    
    # 2. Terapkan logika fallback otomatis (Prioritas mutlak tanpa memicu konflik interaktif)
    if 'phone1' in df_clean.columns:
        # phone1 (Form) -> no_wa_ortu (FO) -> tlp (FO Buku Tamu)
        df_clean['phone1'] = df_clean['phone1'].combine_first(df_clean.get('no_wa_ortu'))
    if 'phone2' in df_clean.columns:
        # phone2 (Form) -> no_wa_anak (FO)
        df_clean['phone2'] = df_clean['phone2'].combine_first(df_clean.get('no_wa_anak'))
        
    # 3. BUANG kolom sumber (FO) agar TIDAK LAGI memicu pop-up bentrok di Tahap 3
    kolom_dibuang = ['no_wa_ortu', 'no_wa_anak']
    df_clean = df_clean.drop(columns=[col for col in kolom_dibuang if col in df_clean.columns])
    
    print(f"✅ HP untuk {nama_tabel} berhasil disatukan ke 'phone1' dan 'phone2'.")
    return df_clean

# === EKSEKUSI PRE-PROCESSING HP ===
table_3_left_join = prep_nomor_hp(table_3_left_join, "Table 3")
table_4_exclusive = prep_nomor_hp(table_4_exclusive, "Table 4")

 🚀 FASE 3 REBORN - TAHAP 2.5: PRE-PROCESSING NOMOR HP 🚀 
✅ HP untuk Table 3 berhasil disatukan ke 'phone1' dan 'phone2'.
✅ HP untuk Table 4 berhasil disatukan ke 'phone1' dan 'phone2'.


In [26]:
import pandas as pd
import numpy as np
import json
import os

print("=================================================================")
print(" 🚀 FASE 3 REBORN - TAHAP 3: RESOLUSI KONFLIK (AUTO-SAVE) 🚀 ")
print("=================================================================")

FILE_MEMORI = 'memori_resolusi.json'

def load_memori():
    if os.path.exists(FILE_MEMORI):
        with open(FILE_MEMORI, 'r') as f:
            return json.load(f)
    return {}

def save_memori(data_memori):
    with open(FILE_MEMORI, 'w') as f:
        json.dump(data_memori, f, indent=4)

def resolusi_konflik_interaktif_autosave(df, nama_tabel):
    print(f"\n🔍 Mulai memindai tabel: {nama_tabel}...")
    df_clean = df.copy()
    memori_manual = load_memori()
    
    # MAPPING KOLOM (Target : List Source)
    mapping_kolom = {
        'name': ['fullName', 'nama_l', 'nama'],
        'nickname': ['nama_p', 'nickName'],
        'sekolah': ['schoolName', 'nama_instansi'],
        'gender': ['gender', 'jenis_k'],
        'kewarganegaraan': ['kewarganegaraan_form', 'kewarganegaraan_admin'],
        'sumber_info': ['info_form', 'info_admin'],
        'bank': ['bank_form', 'bank_admin'],
        'bulan_masuk': ['bulan_masuk_form', 'bulan_masuk_admin'],
        'provinsi': ['provinsi_form', 'provinsi_admin'],
        'kabupaten': ['kabupaten_form', 'kabupaten_admin'],
        'email': ['email_admin', 'email_datautama', 'join_email', 'email_form'],
        'pilihan_program': ['program', 'pilihan_program'],
        'tujuan_program': ['purpose', 'tujuan_program'],
        'kurikulum': ['curriculum', 'kurikulum'],
        'status': ['status_form', 'status_admin'],
        'jenis_test': ['jenis_test_form', 'jenis_test_admin'],
        'tanggal_trial': ['tgl_trial', 'trial_date'],
        'tanggal_pembayaran': ['tanggal_pembayaran', 'tglbayar_date'],
        'created_at': ['created_at_form', 'created_at_admin']
    }

    # PROSES RESOLUSI
    for index, row in df_clean.iterrows():
        kunci_email = str(row.get('join_email', f'Baris_{index}')).strip()
        if kunci_email not in memori_manual: memori_manual[kunci_email] = {}
            
        for target_col, source_cols in mapping_kolom.items():
            nilai_valid = {} 
            for col in source_cols:
                if col in df_clean.columns:
                    val = row.get(col)
                    if pd.notna(val) and str(val).strip() not in ['', '-', 'nan', 'None']:
                        nilai_valid[col] = str(val).strip()
            
            unique_vals = list(set(nilai_valid.values()))
            
            if len(unique_vals) == 1:
                df_clean.at[index, target_col] = unique_vals[0]
            elif len(unique_vals) > 1:
                if target_col in memori_manual[kunci_email]:
                    df_clean.at[index, target_col] = memori_manual[kunci_email][target_col]
                else:
                    print(f"\n⚠️ KONFLIK BARU | Email: {kunci_email} | Kolom: {target_col}")
                    pilihan_list = list(nilai_valid.values())
                    for i, val in enumerate(pilihan_list): print(f"  [{i+1}] {val}")
                    pilihan = input("Pilih angka (0 untuk manual): ")
                    
                    if pilihan.isdigit() and 1 <= int(pilihan) <= len(pilihan_list):
                        terpilih = pilihan_list[int(pilihan)-1]
                    else:
                        terpilih = input("✍️ Ketik nilai manual: ")
                    
                    df_clean.at[index, target_col] = terpilih
                    memori_manual[kunci_email][target_col] = terpilih 
            else:
                df_clean.at[index, target_col] = None

    save_memori(memori_manual)

    # PERBAIKAN DROP KOLOM DENGAN IF-ELSE (Aman dari KeyError)
    for target_col, source_cols in mapping_kolom.items():
        for col in source_cols:
            # Drop hanya jika kolom sumber ada, dan JIKA bukan merupakan kolom target itu sendiri
            if col in df_clean.columns and col != target_col:
                df_clean = df_clean.drop(columns=[col])
    
    print(f"🎉 Pembersihan {nama_tabel} selesai!")
    return df_clean

# === EKSEKUSI ===
table_3_left_join = resolusi_konflik_interaktif_autosave(table_3_left_join, "Table 3")
table_4_exclusive = resolusi_konflik_interaktif_autosave(table_4_exclusive, "Table 4")

 🚀 FASE 3 REBORN - TAHAP 3: RESOLUSI KONFLIK (AUTO-SAVE) 🚀 

🔍 Mulai memindai tabel: Table 3...
🎉 Pembersihan Table 3 selesai!

🔍 Mulai memindai tabel: Table 4...
🎉 Pembersihan Table 4 selesai!


# Daerah

In [27]:
import pandas as pd
import numpy as np

print("=================================================================")
print(" 🚀 FASE 3 REBORN - TAHAP 4: KONVERSI WILAYAH (ALL-IN-ONE) 🚀 ")
print("=================================================================")

def proses_konversi_wilayah(df_target, df_old, df_new, nama_tabel):
    print(f"\n🌍 Memproses Konversi Wilayah: {nama_tabel}...")
    df = df_target.copy()
    
    # 1. BERSINKAN NILAI 0 MENJADI NONE (Mencegah error mapping)
    kolom_wilayah = ['provinsi_final', 'kabupaten_final', 'kecamatan', 'kelurahan']
    for col in kolom_wilayah:
        if col in df.columns:
            kondisi_nol = (df[col] == 0) | (df[col] == 0.0) | (df[col].astype(str).str.strip() == '0') | (df[col].astype(str).str.strip() == '0.0')
            df.loc[kondisi_nol, col] = None

    # 2. PERSIAPAN DATA MASTER KABUPATEN LAMA (HARD-FIX SURABAYA & JAKARTA)
    df_kab_lama = df_old['kabupaten'].copy()
    if 'name' in df_kab_lama.columns:
        # Fix Surabaya
        mask_sby = df_kab_lama['name'].astype(str).str.strip().str.upper() == 'SURABAYA'
        df_kab_lama.loc[mask_sby, 'name'] = 'KOTA SURABAYA'
        
        # Fix Jakarta
        kamus_jkt = {
            'KOTA ADM. JAKARTA PUSAT': 'Kota Administrasi Jakarta Pusat',
            'KOTA ADM. JAKARTA UTARA': 'Kota Administrasi Jakarta Utara',
            'KOTA ADM. JAKARTA BARAT': 'Kota Administrasi Jakarta Barat',
            'KOTA ADM. JAKARTA SELATAN': 'Kota Administrasi Jakarta Selatan',
            'KOTA ADM. JAKARTA TIMUR': 'Kota Administrasi Jakarta Timur'
        }
        df_kab_lama['name'] = df_kab_lama['name'].apply(lambda x: kamus_jkt.get(str(x).strip().upper(), x))

    # 3. BUAT KAMUS JEMBATAN PINTAR (OLD ID -> NAMA -> NEW ID)
    def proses_konversi_wilayah(df_target, df_old, df_new, nama_tabel):
        print(f"\n🌍 Memproses Konversi Wilayah (Update Kolom Spesifik): {nama_tabel}...")
        df = df_target.copy()
    
    # 1. BERSINKAN NILAI 0 MENJADI NONE
    kolom_wilayah = ['provinsi', 'kabupaten', 'kecamatan', 'kelurahan']
    for col in kolom_wilayah:
        if col in df.columns:
            df.loc[(df[col] == 0) | (df[col] == '0'), col] = None

    # 2. BUAT KAMUS JEMBATAN PINTAR (OLD ID -> NEW ID)
    # Kita pakai nama kolom spesifik yang kamu berikan
    def build_map(df_o, df_n, col_id_old, col_name_old, col_id_new, col_name_new):
        map_id_nama = dict(zip(df_o[col_id_old].astype(str).str.strip(), df_o[col_name_old].astype(str).str.strip().str.lower()))
        map_nama_id = dict(zip(df_n[col_name_new].astype(str).str.strip().str.lower(), df_n[col_id_new]))
        return {old_id: map_nama_id.get(nama) for old_id, nama in map_id_nama.items()}

    map_prov = build_map(df_old['provinsi'], df_new['provinsi'], 'idprovinsi', 'nama', 'id_provinsi', 'nama_provinsi')
    map_kab = build_map(df_old['kabupaten'], df_new['kabupaten'], 'idkabupaten', 'name', 'id_kabupaten', 'nama_kabupaten')
    map_kec = build_map(df_old['kecamatan'], df_new['kecamatan'], 'idkecamatan', 'nama', 'id_kecamatan', 'nama_kecamatan')
    map_kel = build_map(df_old['kelurahan'], df_new['kelurahan'], 'idkelurahan', 'nama', 'id_kelurahan', 'nama_kelurahan')

    # 3. TERAPKAN KONVERSI
    def clean_id(val):
        if pd.isna(val): return None
        v = str(val).split('.')[0].strip() # Handle .0 dan spasi
        return v

    if 'provinsi' in df.columns:
        df['id_provinsi_baru'] = df['provinsi'].apply(lambda x: map_prov.get(clean_id(x)))
    if 'kabupaten' in df.columns:
        df['id_kabupaten_baru'] = df['kabupaten'].apply(lambda x: map_kab.get(clean_id(x)))
    if 'kecamatan' in df.columns:
        df['id_kecamatan_baru'] = df['kecamatan'].apply(lambda x: map_kec.get(clean_id(x)))
    if 'kelurahan' in df.columns:
        df['id_kelurahan_baru'] = df['kelurahan'].apply(lambda x: map_kel.get(clean_id(x)))
        
    print(f"✓ Konversi {nama_tabel} sukses!")
    return df

# === EKSEKUSI KONVERSI WILAYAH ===
# Timpa variabel lama dengan DataFrame yang sudah ketambahan kolom ID wilayah baru
table_3_left_join = proses_konversi_wilayah(table_3_left_join, df_old, df_new, "Table 3")
table_4_exclusive = proses_konversi_wilayah(table_4_exclusive, df_old, df_new, "Table 4")

# Pratinjau Hasil Konversi di Table 3
print("\n--- PRATINJAU HASIL MAPPING (OLD ID -> NEW ID) ---")
kolom_pratinjau = ['provinsi', 'id_provinsi_baru', 'kabupaten', 'id_kabupaten_baru']
display(table_3_left_join[kolom_pratinjau])
display(table_4_exclusive[kolom_pratinjau])

 🚀 FASE 3 REBORN - TAHAP 4: KONVERSI WILAYAH (ALL-IN-ONE) 🚀 

🌍 Memproses Konversi Wilayah: Table 3...
✓ Konversi Table 3 sukses!

🌍 Memproses Konversi Wilayah: Table 4...
✓ Konversi Table 4 sukses!

--- PRATINJAU HASIL MAPPING (OLD ID -> NEW ID) ---


,provinsi,id_provinsi_baru,kabupaten,id_kabupaten_baru
0,35,11.0,3578.0,162.0
1,35,11.0,3578.0,162.0
2,35,11.0,3578.0,162.0
3,35,11.0,3578.0,162.0
4,35,11.0,3578.0,162.0
...,...,...,...,...
161,35,11.0,3578.0,162.0
162,35,11.0,3578.0,162.0
163,35,11.0,3578.0,162.0
164,35,11.0,3578.0,162.0


,provinsi,id_provinsi_baru,kabupaten,id_kabupaten_baru
0,None,None,None,None
1,None,None,None,None
2,None,None,None,None
3,None,None,None,None
4,None,None,None,None
5,None,None,None,None
6,None,None,None,None
7,None,None,None,None
8,None,None,None,None
9,None,None,None,None


# Drop column double

In [28]:
# Daftar kolom-kolom asli (sumber) yang sudah kita lebur ke kolom '_final'
# Kolom-kolom ini aman untuk dibuang karena datanya sudah pindah ke kolom '_final'    
kolom_untuk_dibuang = [
    # Identitas
    'fullName', 'nama_l', 'nama', 'nickName', 'nama_p',
    # Kontak
    'no_wa_anak', 'no_wa_ortu', 'email_admin', 'email_datautama', 'join_email', 'email_form',
    # Sekolah
    'schoolName', 'nama_instansi', 'classLevel',
    # Gender
    'jenis_k',
    # Kewarganegaraan
    'kewarganegaraan_form', 'kewarganegaraan_admin',
    # Info
    'info_form', 'info_admin',
    # Bank & Pembayaran
    'bank_form', 'bank_admin',
    # Bulan Masuk
    'bulan_masuk_form', 'bulan_masuk_admin',
    # Test
    'jenis_test_form', 'jenis_test_admin',
    # Wilayah
    'provinsi_form', 'provinsi_admin', 'kabupaten_form', 'kabupaten_admin', 'provinsi', 'kabupaten', 'kecamatan', 'kelurahan',
    # Program & Tujuan
    'program', 'purpose', 
    # Kurikulum
    'curriculum',
    # Status
    'status_form', 'status_admin',
    # Tanggal
    'tgl_trial', 'trial_date', 'tglbayar_date',
    'created_at_form', 'created_at_admin'
]

def drop_old_columns(df, daftar_kolom):
    # Kita hanya drop kolom yang benar-benar ada di dataframe untuk menghindari error
    kolom_yang_ada = [col for col in daftar_kolom if col in df.columns]
    return df.drop(columns=kolom_yang_ada)

# Eksekusi Drop pada kedua tabel
table_3_left_join = drop_old_columns(table_3_left_join, kolom_untuk_dibuang)
table_4_exclusive = drop_old_columns(table_4_exclusive, kolom_untuk_dibuang)

print("✅ Kolom-kolom lama telah berhasil dibuang dari Table 3 dan Table 4.")
print(f"Total kolom Table 3 saat ini: {len(table_3_left_join.columns)}")
print(f"Total kolom Table 4 saat ini: {len(table_4_exclusive.columns)}")

# Audit kolom yang tersisa
print("\n--- Sisa Kolom di Table 3 ---")
display(table_3_left_join)
display(table_4_exclusive)

✅ Kolom-kolom lama telah berhasil dibuang dari Table 3 dan Table 4.
Total kolom Table 3 saat ini: 96
Total kolom Table 4 saat ini: 96

--- Sisa Kolom di Table 3 ---


,idcalon,phone1,phone2,gender,activities,otherActivities,exp,diagnostic,class_options,officeApp,...,bulan_masuk,email,status,jenis_test,tanggal_trial,created_at,id_provinsi_baru,id_kabupaten_baru,id_kecamatan_baru,id_kelurahan_baru
0,C00000017,082231346758,NaN,Perempuan,None,None,sudah pernah,Speaking dan Conversation,offline,None,...,September,puterihapsari.f@gmail.com,1,Tanpa Placement Test,2025-09-15,2025-09-15 13:33:49,11.0,162.0,None,None
1,C00000018,081336647476,NaN,Perempuan,None,None,belum pernah,Iya,offline,None,...,September,dwirohm4@gmail.com,1,A1,2025-09-16,2025-09-16 11:13:41,11.0,162.0,None,None
2,C00000019,085707179656,083849241708,Perempuan,None,None,Tidak,None,online,None,...,None,chyzryth@gmail.com,1,None,None,2025-09-17 17:32:16,11.0,162.0,None,None
3,C00000020,087765283592,NaN,Laki-laki,None,None,belum pernah,Ada,offline,None,...,None,melisnifuku@gmail.com,1,None,2025-09-17,2025-09-17 16:49:20,11.0,162.0,None,None
4,C00000021,081554932188,081554932188,Perempuan,None,None,sudah pernah,Ya,online,None,...,September,afadhilpa@gmail.com,0,None,2025-09-18,2025-09-17 17:16:41,11.0,162.0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,C00000197,08170326911,NaN,Laki-laki,None,None,belum pernah,"Ya, engga paham pertanyaan, tidak bisa baca pe...",offline,None,...,None,adeodatus.kurniawan@gmail.com,0,None,None,2026-04-02 13:20:19,11.0,162.0,None,None
162,C00000198,082244441630,NaN,Perempuan,None,None,sudah pernah,Belum ada,offline,None,...,None,nroskalindha18@gmail.com,1,None,None,2026-04-13 15:58:41,11.0,162.0,None,None
163,C00000199,08977257033,NaN,Perempuan,Mahasiswa,,None,None,offline,None,...,None,clarizarisa3@gmail.com,0,None,None,2026-04-13 17:28:08,11.0,162.0,None,None
164,C00000200,085850209079,NaN,Perempuan,None,None,belum pernah,"Bahasa Inggris hanya dipakai di kelas, kurang ...",offline,None,...,None,shaqilaanindra@gmail.com,1,None,None,2026-04-14 07:20:01,11.0,162.0,None,None


,idcatatanawal_admin,tlp,updated_at,idcatatanawal_datautama,pengontak_admin,pilihan_program,jenis_program,level_1,level_2,tujuan_program,...,bulan_masuk,email,status,jenis_test,tanggal_trial,created_at,id_provinsi_baru,id_kabupaten_baru,id_kecamatan_baru,id_kelurahan_baru
0,M000000001,0812312344531,2025-09-12 00:51:16,NaN,NaN,None,NaN,NaN,NaN,None,...,None,ahmad@gmail.com,canceled,None,None,2025-09-12 00:51:16,None,None,None,None
1,M000000017,089677819546,2025-09-24 14:02:51,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-09-24 14:02:51,None,None,None,None
2,M000000018,081340370088,2025-09-24 14:07:30,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-09-24 14:07:30,None,None,None,None
3,M000000020,085786164620,2025-10-03 11:40:19,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-10-03 11:40:19,None,None,None,None
4,M000000026,081313223427,2025-10-03 12:39:02,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-10-03 12:39:02,None,None,None,None
5,M000000027,081231225959,2025-10-03 12:41:41,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-10-03 12:41:41,None,None,None,None
6,M000000028,085219083605,2025-10-03 15:42:26,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,canceled,None,None,2025-10-03 15:40:27,None,None,None,None
7,M000000029,088217919663,2025-10-03 15:43:23,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-10-03 15:43:23,None,None,None,None
8,M000000030,085755891737,2025-10-03 15:44:18,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-10-03 15:44:18,None,None,None,None
9,M000000031,085706406904,2025-10-03 15:45:13,NaN,NaN,None,NaN,NaN,NaN,None,...,None,None,follow up another time,None,None,2025-10-03 15:45:13,None,None,None,None


In [29]:
# 1. Mendapatkan daftar kolom dari kedua tabel
kolom_t3 = set(table_3_left_join.columns)
kolom_t4 = set(table_4_exclusive.columns)

# 2. Cek apakah jumlah kolomnya sama
print(f"Total kolom Table 3: {len(kolom_t3)}")
print(f"Total kolom Table 4: {len(kolom_t4)}")

# 3. Cari kolom yang ada di T3 tapi tidak ada di T4
beda_t3_ke_t4 = kolom_t3 - kolom_t4
# 4. Cari kolom yang ada di T4 tapi tidak ada di T3
beda_t4_ke_t3 = kolom_t4 - kolom_t3

if not beda_t3_ke_t4 and not beda_t4_ke_t3:
    print("\n✅ Mantap! Nama kolom di kedua tabel sudah IDENTIK (sama persis).")
else:
    print("\n⚠️ ADA PERBEDAAN NAMA KOLOM:")
    if beda_t3_ke_t4:
        print(f"Kolom di Table 3 tapi tidak ada di Table 4: {beda_t3_ke_t4}")
    if beda_t4_ke_t3:
        print(f"Kolom di Table 4 tapi tidak ada di Table 3: {beda_t4_ke_t3}")
        
display(table_3_left_join.info())
display(table_4_exclusive.info())

Total kolom Table 3: 96
Total kolom Table 4: 96

✅ Mantap! Nama kolom di kedua tabel sudah IDENTIK (sama persis).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 96 columns):
 #   Column                    Non-Null Count  Dtype          
---  ------                    --------------  -----          
 0   idcalon                   166 non-null    object         
 1   phone1                    166 non-null    object         
 2   phone2                    7 non-null      object         
 3   gender                    160 non-null    object         
 4   activities                62 non-null     object         
 5   otherActivities           62 non-null     object         
 6   exp                       104 non-null    object         
 7   diagnostic                97 non-null     object         
 8   class_options             166 non-null    object         
 9   officeApp                 7 non-null      object         
 10  editing             

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 96 columns):
 #   Column                    Non-Null Count  Dtype          
---  ------                    --------------  -----          
 0   idcatatanawal_admin       28 non-null     object         
 1   tlp                       28 non-null     object         
 2   updated_at                28 non-null     datetime64[ns] 
 3   idcatatanawal_datautama   0 non-null      object         
 4   pengontak_admin           0 non-null      object         
 5   pilihan_program           0 non-null      object         
 6   jenis_program             0 non-null      object         
 7   level_1                   0 non-null      object         
 8   level_2                   0 non-null      object         
 9   tujuan_program            0 non-null      object         
 10  metode                    0 non-null      object         
 11  referensi                 0 non-null      object         
 12  sby_luarsb

None

In [ ]:
# 1. Tentukan urutan kolom yang ideal (kelompokkan berdasarkan kategori)
urutan_kolom = [
    # Identitas Utama
    'idcalon', 'name', 'nickname', 'gender', 'kewarganegaraan', 'email',
    
    # Kontak & Ortu
    'phone1', 'phone2', 'tlp', 'nama_ortu', 'pekerjaan_ortu',
    
    # Pendidikan & Sekolah
    'sekolah', 'pernah_les', 'kesulitan_pelajaran', 'kurikulum', 'exp', 'activities', 'otherActivities',
    
    # Wilayah (Baru & Lama)
    'id_provinsi_baru', 'id_kabupaten_baru', 'id_kecamatan_baru', 'id_kelurahan_baru', 'alamat_lengkap',
    
    # Pendaftaran & Program
    'sumber_info', 'pilihan_program', 'jenis_program', 'tujuan_program', 'class_options', 'bank', 'bulan_masuk',
    
    # Histori Proses / Sesi
    'created_at', 'tanggal_trial', 'waktu_test1', 'waktu_test2', 'jenis_test', 'hasil_test', 'wawancara', 'diterima_dikelas',
    
    # ID & Metadata Sistem
    'idcatatanawal_admin', 'idcatatanawal_datautama', 'idpendkursus', 'status'
]

# 2. Tambahkan kolom yang mungkin belum masuk di daftar di atas (agar tidak hilang)
kolom_tersisa = [col for col in table_3_left_join.columns if col not in urutan_kolom]
urutan_final = urutan_kolom + kolom_tersisa

# 3. Terapkan urutan baru ke DataFrame
table_3_left_join = table_3_left_join[urutan_final]
table_4_exclusive = table_4_exclusive[urutan_final]

print("✅ Urutan kolom berhasil ditata ulang agar lebih enak dilihat!")
display(table_3_left_join)
display(table_4_exclusive)

✅ Urutan kolom berhasil ditata ulang agar lebih enak dilihat!


,idcalon,name,nickname,gender,kewarganegaraan,email,phone1,phone2,tlp,nama_ortu,...,trial,catatan_penting,form_daftar,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,tglmasuk_date,tglkeluar_date
0,C00000017,Daria Azmiya Jasmine,Daria,Perempuan,Indonesia,puterihapsari.f@gmail.com,082231346758,NaN,082231346758,,...,,,Belum,,,O000000009,2025-09-15,None,2025-09-24,None
1,C00000018,Annisa Zahro Ramadhania,Annisa,Perempuan,Indonesia,dwirohm4@gmail.com,081336647476,NaN,081336647476,,...,,,Belum,,,O000000014,2025-09-16,2025-09-16,2025-09-25,None
2,C00000019,Zulfa Bariatur Rahma,Zulfa,Perempuan,Indonesia,chyzryth@gmail.com,085707179656,083849241708,083849241708,,...,,,Belum,,,O000000007,2025-09-16,None,None,None
3,C00000020,Achmad Naufal Albiruni,Albi,Laki-laki,Indonesia,melisnifuku@gmail.com,087765283592,NaN,087765283592,NaN,...,,,Belum,,,O000000013,2025-09-17,2025-09-17,2025-09-25,None
4,C00000021,Khansa Amalia Putri Aji,Khansa,Perempuan,Indonesia,afadhilpa@gmail.com,081554932188,081554932188,081554932188,,...,,,Belum,,,O000000017,2025-09-15,2025-09-18,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,C00000197,Antonius Miguel Kurniawan,Miguel,Laki-laki,Indonesia,adeodatus.kurniawan@gmail.com,08170326911,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,C00000198,Tisha kayla janitra,Tisha,Perempuan,Indonesia,nroskalindha18@gmail.com,082244441630,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163,C00000199,Clariza Arifianti,Clara,Perempuan,Indonesia,clarizarisa3@gmail.com,08977257033,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
164,C00000200,Shaqila Anindra Dzakira,Shaqila,Perempuan,Indonesia,shaqilaanindra@gmail.com,085850209079,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,idcalon,name,nickname,gender,kewarganegaraan,email,phone1,phone2,tlp,nama_ortu,...,trial,catatan_penting,form_daftar,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,tglmasuk_date,tglkeluar_date
0,NaN,Ahmad 1,None,None,None,ahmad@gmail.com,NaN,NaN,0812312344531,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
1,NaN,Bu Tika,None,None,None,None,NaN,NaN,089677819546,NaN,...,,,Belum,,,O000000018,2025-09-17,None,None,None
2,NaN,Bu Devi,None,None,None,None,NaN,NaN,081340370088,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
3,NaN,Jenah,None,None,None,None,NaN,NaN,085786164620,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
4,NaN,Inna,None,None,None,None,NaN,NaN,081313223427,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
5,NaN,Feb,None,None,None,None,NaN,NaN,081231225959,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
6,NaN,Intan,None,None,None,None,NaN,NaN,085219083605,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
7,NaN,Herni,None,None,None,None,NaN,NaN,088217919663,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
8,NaN,Sisca,None,None,None,None,NaN,NaN,085755891737,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN
9,NaN,Tilla,None,None,None,None,NaN,NaN,085706406904,NaN,...,,,Belum,,,NaN,NaN,NaN,NaN,NaN


In [31]:
import pandas as pd
import numpy as np

def final_polish_data(df):
    print("✨ Memulai proses pembersihan akhir (Polishing)...")
    df_polished = df.copy()
    
    # 1. DAFTAR KOLOM YANG TIDAK BOLEH DISENTUH (TETAP STRING/OBJECT)
    # Masukkan semua kolom ID, Telepon, dan Kode ke sini
    kolom_kecualikan = [
        'idcalon', 'id_kontak_prospek', 'kode_kontak', 'phone1', 'phone2', 
        'email', 'idpendkursus', 'idcatatanawal_admin', 'nomor_invoice'
    ]
    
    # 2. SAPU BERSIH DATA KOTOR
    df_polished = df_polished.replace([r'^\s*$', '-', 'nan', 'None', 'NULL'], np.nan, regex=True)
    
    # 3. KONVERSI ANGKA YANG BENAR-BENAR ANGKA (FLOT -> INT)
    for col in df_polished.columns:
        # Pengecualian ketat
        if 'date' in col.lower() or 'created_at' in col.lower() or 'updated_at' in col.lower():
            continue
        if col in kolom_kecualikan:
            continue
            
        # Coba ubah ke numerik
        try:
            # Gunakan pd.to_numeric dengan errors='coerce' agar yang bukan angka jadi NaN
            temp_col = pd.to_numeric(df_polished[col], errors='coerce')
            
            # Jika kolom tersebut berhasil jadi numerik (tidak semua NaN)
            if temp_col.notna().any():
                # Pastikan ini memang kolom angka, bukan ID yang terdeteksi secara tidak sengaja
                # Kita ubah hanya jika tipe aslinya float dan tidak termasuk daftar pengecualian
                if df_polished[col].dtype in ['float64', 'float32']:
                    df_polished[col] = temp_col.fillna(-1).astype(int).replace(-1, np.nan)
        except:
            continue
            
    # 4. MENGUBAH NaN MENJADI None (Agar ramah MySQL)
    df_polished = df_polished.astype(object).where(pd.notnull(df_polished), None)
    
    print("✓ Pembersihan mutlak selesai! Data sudah bersih tanpa merusak format HP/ID.")
    return df_polished

# === EKSEKUSI PADA KEDUA TABEL ===
table_3_left_join = final_polish_data(table_3_left_join)
table_4_exclusive = final_polish_data(table_4_exclusive)

print("\n--- PRATINJAU HASIL (Pastikan HP tidak ada .0) ---")
display(table_3_left_join)

✨ Memulai proses pembersihan akhir (Polishing)...
✓ Pembersihan mutlak selesai! Data sudah bersih tanpa merusak format HP/ID.
✨ Memulai proses pembersihan akhir (Polishing)...
✓ Pembersihan mutlak selesai! Data sudah bersih tanpa merusak format HP/ID.

--- PRATINJAU HASIL (Pastikan HP tidak ada .0) ---


,idcalon,name,nickname,gender,kewarganegaraan,email,phone1,phone2,tlp,nama_ortu,...,trial,catatan_penting,form_daftar,wag_lv,pic,idcatatanawal_tglpenting,kontakA_date,wawancara_date,tglmasuk_date,tglkeluar_date
0,C00000017,Daria Azmiya Jasmine,Daria,Perempuan,Indonesia,puterihapsari.f@gmail.com,082231346758,None,082231346758,None,...,None,None,Belum,None,None,O000000009,2025-09-15,None,2025-09-24,None
1,C00000018,Annisa Zahro Ramadhania,Annisa,Perempuan,Indonesia,dwirohm4@gmail.com,081336647476,None,081336647476,None,...,None,None,Belum,None,None,O000000014,2025-09-16,2025-09-16,2025-09-25,None
2,C00000019,Zulfa Bariatur Rahma,Zulfa,Perempuan,Indonesia,chyzryth@gmail.com,085707179656,083849241708,083849241708,None,...,None,None,Belum,None,None,O000000007,2025-09-16,None,None,None
3,C00000020,Achmad Naufal Albiruni,Albi,None,Indonesia,melisnifuku@gmail.com,087765283592,None,087765283592,None,...,None,None,Belum,None,None,O000000013,2025-09-17,2025-09-17,2025-09-25,None
4,C00000021,Khansa Amalia Putri Aji,Khansa,Perempuan,Indonesia,afadhilpa@gmail.com,081554932188,081554932188,081554932188,None,...,None,None,Belum,None,None,O000000017,2025-09-15,2025-09-18,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,C00000197,Antonius Miguel Kurniawan,Miguel,None,Indonesia,adeodatus.kurniawan@gmail.com,08170326911,None,None,None,...,None,None,None,None,None,None,None,None,None,None
162,C00000198,Tisha kayla janitra,Tisha,Perempuan,Indonesia,nroskalindha18@gmail.com,082244441630,None,None,None,...,None,None,None,None,None,None,None,None,None,None
163,C00000199,Clariza Arifianti,Clara,Perempuan,Indonesia,clarizarisa3@gmail.com,08977257033,None,None,None,...,None,None,None,None,None,None,None,None,None,None
164,C00000200,Shaqila Anindra Dzakira,Shaqila,Perempuan,Indonesia,shaqilaanindra@gmail.com,085850209079,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [32]:
# display(table_3_left_join)
# display(table_3_left_join.info())
# display(table_4_exclusive)
# display(table_4_exclusive.info())

# Pindahan kontak prospek

In [33]:
import pandas as pd
import random
import string

def generate_kode_kontak(length=8):
    """Membuat kode unik acak (Huruf Besar & Angka)."""
    chars = string.ascii_uppercase + string.digits
    return ''.join(random.choices(chars, k=length))

def proses_pindah_ke_kontak_prospek(table_3, table_4):
    print("🚀 Memulai proses migrasi ke: kontak_prospek...")
    
    # Gabungkan semua data dari T3 dan T4
    df_source = pd.concat([table_3, table_4], ignore_index=True)
    
    # Buat DataFrame target sesuai struktur
    df_target = pd.DataFrame()
    
    # --- 1. MAPPING DATA ---
    df_target['nama_penanya'] = df_source.get('name', None)
    df_target['nomor_telepon'] = df_source.get('phone1', None)
    df_target['email'] = df_source.get('email', None)
    df_target['sumber_informasi'] = df_source.get('sumber_info', None)
    df_target['catatan_awal_fo'] = df_source.get('catatan_admin', None)
    df_target['status_kontak'] = df_source.get('status', 'prospek')
    df_target['id_admin_fo'] = df_source.get('pic', None) # Asumsi PIC admin FO
    df_target['tanggal_kontak_pertama'] = df_source.get('created_at', None)
    df_target['tanggal_kontak_terakhir'] = df_source.get('updated_at', df_source.get('created_at'))
    
    # --- 2. GENERATE ID & KODE (Looping & Random) ---
    
    # Looping Int untuk id_kontak_prospek (1, 2, 3...)
    df_target['id_kontak_prospek'] = range(1, len(df_target) + 1)
    
    # Kode Kontak Random (Huruf Besar & Angka)
    df_target['kode_kontak'] = [generate_kode_kontak() for _ in range(len(df_target))]
    
    # --- 3. FINAL CLEANUP ---
    # Pastikan tipe data rapi
    df_target['id_kontak_prospek'] = df_target['id_kontak_prospek'].astype(int)
    
    # Urutkan kolom sesuai struktur yang kamu inginkan
    urutan_kolom = [
        'id_kontak_prospek', 'kode_kontak', 'nama_penanya', 'nomor_telepon', 
        'email', 'sumber_informasi', 'catatan_awal_fo', 'id_admin_fo', 
        'status_kontak', 'tanggal_kontak_pertama', 'tanggal_kontak_terakhir'
    ]
    df_target = df_target[urutan_kolom]
    
    print(f"✅ Berhasil memproses {len(df_target)} data ke 'kontak_prospek'!")
    return df_target

# === EKSEKUSI ===
df_future['kontak_prospek'] = proses_pindah_ke_kontak_prospek(table_3_left_join, table_4_exclusive)

display(df_future['kontak_prospek'])
display(df_future['kontak_prospek'].info())

🚀 Memulai proses migrasi ke: kontak_prospek...
✅ Berhasil memproses 194 data ke 'kontak_prospek'!


,id_kontak_prospek,kode_kontak,nama_penanya,nomor_telepon,email,sumber_informasi,catatan_awal_fo,id_admin_fo,status_kontak,tanggal_kontak_pertama,tanggal_kontak_terakhir
0,1,OVNZAR0I,Daria Azmiya Jasmine,082231346758,puterihapsari.f@gmail.com,Teman/kerabat/saudara,None,None,1,None,2025-09-18 17:39:40
1,2,RJX87IO7,Annisa Zahro Ramadhania,081336647476,dwirohm4@gmail.com,Instagram,None,None,1,None,2025-09-23 16:25:14
2,3,DRKGTC3T,Zulfa Bariatur Rahma,085707179656,chyzryth@gmail.com,Lainnya,None,None,1,None,2025-10-03 09:48:34
3,4,U42DMFNE,Achmad Naufal Albiruni,087765283592,melisnifuku@gmail.com,Instagram,None,None,1,None,2025-10-03 09:49:15
4,5,9BSR9RDW,Khansa Amalia Putri Aji,081554932188,afadhilpa@gmail.com,Teman/kerabat/saudara,None,None,0,None,2025-10-03 09:48:50
...,...,...,...,...,...,...,...,...,...,...,...
189,190,8L500RRT,Atiya Sabita,None,None,None,None,None,waiting for confirmation,None,2025-10-17 15:41:09
190,191,EAJQXKRN,Dwi,None,None,None,None,None,follow up another time,None,2025-10-17 15:41:26
191,192,53QUX8MQ,Aca,None,None,None,None,None,follow up another time,None,2025-10-24 15:27:54
192,193,TTSY0WIZ,Bu Hartik,None,None,None,None,None,waiting for confirmation,None,2025-10-24 15:28:21


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   id_kontak_prospek        194 non-null    int64 
 1   kode_kontak              194 non-null    object
 2   nama_penanya             186 non-null    object
 3   nomor_telepon            159 non-null    object
 4   email                    163 non-null    object
 5   sumber_informasi         165 non-null    object
 6   catatan_awal_fo          0 non-null      object
 7   id_admin_fo              0 non-null      object
 8   status_kontak            194 non-null    object
 9   tanggal_kontak_pertama   0 non-null      object
 10  tanggal_kontak_terakhir  65 non-null     object
dtypes: int64(1), object(10)
memory usage: 16.8+ KB


None

# calon_siswa

In [34]:
def proses_pindah_ke_calon_siswa(df_source, df_kontak):
    print("🚀 Memulai migrasi data ke: calon_siswa (Struktur Baru)...")
    
    # 1. PERSIAPAN DATA KONT_PROSPEK (Untuk JOIN)
    # Kita siapkan subset data untuk di-join agar tidak duplikat
    df_kontak_subset = df_kontak[['id_kontak_prospek', 'kode_kontak', 'nama_penanya']].copy()
    # Bersihkan nama untuk join
    df_kontak_subset['nama_clean'] = df_kontak_subset['nama_penanya'].astype(str).str.lower().str.strip()
    
    # Ambil baris pertama jika ada nama yang kembar (mengatasi duplikat)
    df_kontak_subset = df_kontak_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    # 2. PROSES PEMETAAN DATA
    df_target = pd.DataFrame()
    df_source['nama_clean'] = df_source['name'].astype(str).str.lower().str.strip()
    
    # JOIN data dari kontak_prospek berdasarkan nama
    df_merged = df_source.merge(
        df_kontak_subset, 
        on='nama_clean', 
        how='left'
    )
    
    # 3. MAPPING KOLOM KE STRUKTUR 34 KOLOM
    df_target['id_calon'] = range(1, len(df_merged) + 1) # Auto-increment
    df_target['kode_unik'] = df_merged['kode_kontak']
    df_target['nama_lengkap'] = df_merged['name']
    df_target['id_kontak_prospek'] = df_merged['id_kontak_prospek']
    df_target['nama_panggilan'] = df_merged['nickname']
    df_target['jenis_kelamin'] = df_merged['gender']
    df_target['kewarganegaraan'] = df_merged['kewarganegaraan']
    df_target['email'] = df_merged['email']
    
    df_target['nama_kontak_awal'] = df_merged['pengontak_admin']
    df_target['wa_kontak_awal'] = df_merged['tlp']
    df_target['assigned_fo'] = df_merged['pic']
    df_target['assigned_akademik'] = None # Belum ada data akademik, jadi None
    df_target['fo_status'] = df_merged['status']    
    
    # Wilayah (sesuaikan kolom hasil konversi wilayah)
    df_target['id_provinsi'] = df_merged.get('id_provinsi_baru')
    df_target['id_kabupaten'] = df_merged.get('id_kabupaten_baru')
    df_target['id_kecamatan'] = df_merged.get('id_kecamatan_baru')
    df_target['id_kelurahan'] = df_merged.get('id_kelurahan_baru')
    df_target['alamat_lengkap'] = df_merged.get('alamat_lengkap')
    
    # Kontak & Lainnya
    df_target['wa_siswa'] = df_merged.get('phone2')
    df_target['wa_ortu'] = df_merged.get('phone1')
    df_target['catatan_awal_fo'] = df_merged.get('catatan_admin')
    
    # Timestamps & Tracking
    df_target['created_at'] = df_merged.get('created_at')
    df_target['updated_at'] = df_merged.get('updated_at')

    # 4. ISI KOLOM KOSONG DENGAN NONE (Memastikan total 34 kolom)
    target_columns = [
        'id_calon', 'kode_unik', 'nama_lengkap', 'id_kontak_prospek', 'nama_panggilan', 
        'jenis_kelamin', 'tempat_lahir', 'tanggal_lahir', 'kewarganegaraan', 'email', 
        'agama', 'nama_kontak_awal', 'wa_kontak_awal', 'id_provinsi', 'id_kabupaten', 
        'id_kecamatan', 'id_kelurahan', 'alamat_lengkap', 'wa_siswa', 'wa_ortu', 
        'wa_administrasi', 'assigned_fo', 'assigned_akademik', 'catatan_awal_fo', 
        'fo_status', 'fo_status_updated_at', 'handover_at', 'link_form_sent_at', 
        'form_completed_at', 'first_submitted_at', 'latest_submitted_at', 
        'deleted_at', 'created_at', 'updated_at'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    # Urutkan sesuai urutan resmi
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
df_future['calon_siswa'] = proses_pindah_ke_calon_siswa(table_3_left_join, df_future['kontak_prospek'])

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA MASTER ---")
display(df_future['calon_siswa'])

🚀 Memulai migrasi data ke: calon_siswa (Struktur Baru)...
✅ Migrasi ke 'calon_siswa' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA MASTER ---


,id_calon,kode_unik,nama_lengkap,id_kontak_prospek,nama_panggilan,jenis_kelamin,tempat_lahir,tanggal_lahir,kewarganegaraan,email,...,fo_status,fo_status_updated_at,handover_at,link_form_sent_at,form_completed_at,first_submitted_at,latest_submitted_at,deleted_at,created_at,updated_at
0,1,OVNZAR0I,Daria Azmiya Jasmine,1,Daria,Perempuan,None,None,Indonesia,puterihapsari.f@gmail.com,...,1,None,None,None,None,None,None,None,None,2025-09-18 17:39:40
1,2,RJX87IO7,Annisa Zahro Ramadhania,2,Annisa,Perempuan,None,None,Indonesia,dwirohm4@gmail.com,...,1,None,None,None,None,None,None,None,None,2025-09-23 16:25:14
2,3,DRKGTC3T,Zulfa Bariatur Rahma,3,Zulfa,Perempuan,None,None,Indonesia,chyzryth@gmail.com,...,1,None,None,None,None,None,None,None,None,2025-10-03 09:48:34
3,4,U42DMFNE,Achmad Naufal Albiruni,4,Albi,None,None,None,Indonesia,melisnifuku@gmail.com,...,1,None,None,None,None,None,None,None,None,2025-10-03 09:49:15
4,5,9BSR9RDW,Khansa Amalia Putri Aji,5,Khansa,Perempuan,None,None,Indonesia,afadhilpa@gmail.com,...,0,None,None,None,None,None,None,None,None,2025-10-03 09:48:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,162,A598A870,Antonius Miguel Kurniawan,162,Miguel,None,None,None,Indonesia,adeodatus.kurniawan@gmail.com,...,0,None,None,None,None,None,None,None,None,None
162,163,GKB3Q73O,Tisha kayla janitra,163,Tisha,Perempuan,None,None,Indonesia,nroskalindha18@gmail.com,...,1,None,None,None,None,None,None,None,None,None
163,164,FUAQGMVP,Clariza Arifianti,164,Clara,Perempuan,None,None,Indonesia,clarizarisa3@gmail.com,...,0,None,None,None,None,None,None,None,None,None
164,165,YNQ26Z8D,Shaqila Anindra Dzakira,165,Shaqila,Perempuan,None,None,Indonesia,shaqilaanindra@gmail.com,...,1,None,None,None,None,None,None,None,None,None


# calon siswa akademik

In [37]:
import pandas as pd

def proses_pindah_ke_calon_siswa_akademik(df_source, df_calon):
    print("🚀 Memulai migrasi data ke: calon_siswa_akademik...")
    
    # 1. PERSIAPAN PENGAMBILAN 'id_calon' DARI TABEL CALON_SISWA
    df_calon_subset = df_calon[['id_calon', 'nama_lengkap']].copy()
    df_calon_subset['nama_clean'] = df_calon_subset['nama_lengkap'].astype(str).str.lower().str.strip()
    # Drop duplikat nama untuk join yang aman
    df_calon_subset = df_calon_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    # 2. PROSES PEMETAAN DATA
    df_target = pd.DataFrame()
    df_source['nama_clean'] = df_source['name'].astype(str).str.lower().str.strip()
    
    # JOIN data dari calon_siswa untuk mendapatkan id_calon yang valid
    df_merged = df_source.merge(
        df_calon_subset, 
        on='nama_clean', 
        how='left'
    )
    
    # 3. MAPPING KOLOM
    df_target['id_calon_akademik'] = range(1, len(df_merged) + 1) # ID Auto-increment
    df_target['id_calon'] = df_merged['id_calon'] # Hasil join
    
    # Data Sekolah
    df_target['nama_sekolah'] = df_merged.get('sekolah')
    df_target['jenjang_kelas_1'] = df_merged.get('level_1') # Asumsi dari pilihan kelas
    df_target['jenjang_kelas_2'] = df_merged.get('level_2')       # Jika ada level 2
    df_target['kurikulum_sekolah'] = df_merged.get('kurikulum')
    
    # Program & Level (Sementara menggunakan nama program/string, nanti bisa diubah jadi ID relasi jika perlu)
    df_target['id_kursus'] = df_merged.get('idpendkursus') 
    df_target['id_periode'] = None # Belum ada data periode, jadi None
    df_target['id_level'] = None # Belum ada data level, jadi None
    df_target['submission_state'] = None # Belum ada data state, jadi None
    df_target['preferensi_metode_belajar'] = df_merged.get('metode')
    
    # Kuesioner Akademik & Aktivitas
    df_target['riwayat_les'] = df_merged.get('pernah_les')
    df_target['kesulitan_belajar'] = df_merged.get('kesulitan_pelajaran')
    df_target['kegiatan_sekarang'] = df_merged.get('activities')
    df_target['kegiatan_lainnya'] = df_merged.get('otherActivities')
    
    # Skill & Hardware
    df_target['kemampuan_officeApp'] = df_merged.get('officeApp')
    df_target['kemampuan_editing'] = df_merged.get('editing')
    df_target['kemampuan_kustom'] = df_merged.get('custom')
    df_target['kemampuan_komputer'] = df_merged.get('computer')
    df_target['kemampuan_software'] = df_merged.get('software')
    df_target['penggunaan_gadget'] = df_merged.get('gadget')
    
    # Informasi & Tujuan
    df_target['sumber_info'] = df_merged.get('sumber_info')
    df_target['referensi'] = df_merged.get('referensi')
    df_target['alasan_daftar'] = df_merged.get('tujuan_program')
    df_target['alasan_program'] = None # Belum ada data alasan program, jadi None
    df_target['harapan_program'] = df_merged.get('hope')
    df_target['lampiran_file'] = df_merged.get('file')
    df_target['submitted_at'] = df_merged.get('created_at')

    # 4. ISI KOLOM KOSONG DENGAN NONE (Memastikan total 28 kolom)
    target_columns = [
        'id_calon_akademik', 'id_calon', 'nama_sekolah', 'jenjang_kelas_1', 
        'jenjang_kelas_2', 'kurikulum_sekolah', 'id_kursus', 'id_periode', 
        'id_level', 'submission_state', 'preferensi_metode_belajar', 
        'riwayat_les', 'kesulitan_belajar', 'kegiatan_sekarang', 'kegiatan_lainnya', 
        'kemampuan_officeApp', 'kemampuan_editing', 'kemampuan_kustom', 
        'kemampuan_komputer', 'kemampuan_software', 'penggunaan_gadget', 
        'sumber_info', 'referensi', 'alasan_daftar', 'alasan_program', 
        'harapan_program', 'lampiran_file', 'submitted_at'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    # Urutkan sesuai urutan resmi
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa_akademik' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
# Panggil fungsi menggunakan table_3_left_join dan df_future['calon_siswa']
df_future['calon_siswa_akademik'] = proses_pindah_ke_calon_siswa_akademik(table_3_left_join, df_future['calon_siswa'])

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA_AKADEMIK MASTER ---")
display(df_future['calon_siswa_akademik'].head())

🚀 Memulai migrasi data ke: calon_siswa_akademik...
✅ Migrasi ke 'calon_siswa_akademik' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA_AKADEMIK MASTER ---


,id_calon_akademik,id_calon,nama_sekolah,jenjang_kelas_1,jenjang_kelas_2,kurikulum_sekolah,id_kursus,id_periode,id_level,submission_state,...,kemampuan_komputer,kemampuan_software,penggunaan_gadget,sumber_info,referensi,alasan_daftar,alasan_program,harapan_program,lampiran_file,submitted_at
0,1,1,TK Al Maghfirah,TK,B,Nasional,K00010,None,None,None,...,None,None,None,Teman/kerabat/saudara,None,None,None,None,None,None
1,2,2,SD Khadijah Wonorejo,SD,3,Cambridge,K00010,None,None,None,...,None,None,None,Instagram,None,supaya lebih paham bhs Inggris,None,None,None,None
2,3,3,SMAN 17 Surabaya,SMA/SMK,10,NASIONAL,K00014,None,None,None,...,sudah pernah,Acode,"Handphone,Laptop",Lainnya,None,UPSKILLING,None,None,None,None
3,4,4,SD Khadijah Wonorejo,SD,5,Cambridge,K00010,None,None,None,...,None,None,None,Instagram,None,None,None,None,None,None
4,5,5,SMAN 17 Surabaya,SMA/SMK,11,Nasional,K00010,None,None,None,...,None,None,None,Teman/kerabat/saudara,None,None,None,None,None,None


# calon siswa ortu

In [41]:
import pandas as pd

def proses_pindah_ke_calon_siswa_ortu(df_source, df_calon):
    print("🚀 Memulai migrasi data ke: calon_siswa_ortu...")
    
    # 1. PERSIAPAN PENGAMBILAN 'id_calon' DARI TABEL CALON_SISWA
    df_calon_subset = df_calon[['id_calon', 'nama_lengkap']].copy()
    df_calon_subset['nama_clean'] = df_calon_subset['nama_lengkap'].astype(str).str.lower().str.strip()
    
    # Drop duplikat nama untuk join yang aman
    df_calon_subset = df_calon_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    # 2. PROSES PEMETAAN DATA
    df_target = pd.DataFrame()
    df_source['nama_clean'] = df_source['name'].astype(str).str.lower().str.strip()
    
    # JOIN data dari calon_siswa untuk mendapatkan id_calon yang valid
    df_merged = df_source.merge(
        df_calon_subset, 
        on='nama_clean', 
        how='left'
    )
    
    # 3. MAPPING KOLOM
    df_target['id_calon_ortu'] = range(1, len(df_merged) + 1) # ID Auto-increment
    df_target['id_calon'] = df_merged['id_calon'] # Hasil join
    
    # Karena di database lama hanya ada "ortu" secara general, kita petakan ke "ayah" sementara
    df_target['nama_wali'] = df_merged.get('nama_ortu')
    df_target['pekerjaan_wali'] = df_merged.get('pekerjaan_ortu')
    df_target['tempat_lahir_wali'] = df_merged.get('tempat_lahir')
    df_target['tanggal_lahir_wali'] = df_merged.get('tanggal_lahir')

    # 4. ISI KOLOM KOSONG DENGAN NONE (Memastikan total 20 kolom)
    target_columns = [
        'id_calon_ortu', 'id_calon', 'nama_ayah', 'pekerjaan_ayah', 
        'pendidikan_ayah', 'penghasilan_ayah', 'tempat_lahir_ayah', 'tanggal_lahir_ayah', 
        'nama_ibu', 'pekerjaan_ibu', 'pendidikan_ibu', 'penghasilan_ibu', 
        'tempat_lahir_ibu', 'tanggal_lahir_ibu', 'nama_wali', 'pekerjaan_wali', 
        'pendidikan_wali', 'penghasilan_wali', 'tempat_lahir_wali', 'tanggal_lahir_wali'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    # Urutkan sesuai urutan resmi
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa_ortu' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
# Panggil fungsi menggunakan table_3_left_join dan df_future['calon_siswa']
df_future['calon_siswa_ortu'] = proses_pindah_ke_calon_siswa_ortu(table_3_left_join, df_future['calon_siswa'])

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA_ORTU MASTER ---")
display(df_future['calon_siswa_ortu'])

🚀 Memulai migrasi data ke: calon_siswa_ortu...
✅ Migrasi ke 'calon_siswa_ortu' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA_ORTU MASTER ---


,id_calon_ortu,id_calon,nama_ayah,pekerjaan_ayah,pendidikan_ayah,penghasilan_ayah,tempat_lahir_ayah,tanggal_lahir_ayah,nama_ibu,pekerjaan_ibu,pendidikan_ibu,penghasilan_ibu,tempat_lahir_ibu,tanggal_lahir_ibu,nama_wali,pekerjaan_wali,pendidikan_wali,penghasilan_wali,tempat_lahir_wali,tanggal_lahir_wali
0,1,1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,2,2,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,3,3,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,4,4,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,5,5,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,162,162,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
162,163,163,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
163,164,164,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
164,165,165,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


# CALON_SISWA_JADWAL

In [43]:
import pandas as pd

def proses_pindah_ke_calon_siswa_jadwal(df_source, df_calon, df_akademik):
    print("🚀 Memulai migrasi data ke: calon_siswa_jadwal...")
    
    # 1. PERSIAPAN JEMBATAN ID (Mencari id_calon_akademik)
    # A. Ambil nama dan id_calon dari tabel calon_siswa
    df_calon_subset = df_calon[['id_calon', 'nama_lengkap']].copy()
    df_calon_subset['nama_clean'] = df_calon_subset['nama_lengkap'].astype(str).str.lower().str.strip()
    df_calon_subset = df_calon_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    # B. Ambil id_calon dan id_calon_akademik dari tabel calon_siswa_akademik
    df_akademik_subset = df_akademik[['id_calon_akademik', 'id_calon']].copy()
    df_akademik_subset = df_akademik_subset.drop_duplicates(subset=['id_calon'], keep='first')
    
    # 2. PROSES JOIN BERANTAI
    df_merged = df_source.copy()
    df_merged['nama_clean'] = df_merged['name'].astype(str).str.lower().str.strip()
    
    # Join 1: Dapatkan id_calon
    df_merged = df_merged.merge(df_calon_subset[['nama_clean', 'id_calon']], on='nama_clean', how='left')
    
    # Join 2: Dapatkan id_calon_akademik menggunakan id_calon
    df_merged = df_merged.merge(df_akademik_subset, on='id_calon', how='left')
    
    # 3. MAPPING KOLOM TANGGAL
    df_target = pd.DataFrame()
    df_target['id_calon_jadwal'] = range(1, len(df_merged) + 1) # ID Auto-increment
    df_target['id_calon_akademik'] = df_merged['id_calon_akademik'] # Hasil join jembatan
    
    # Pemetaan tanggal (sesuaikan dengan nama kolom di table_3 kamu)
    df_target['tanggal_kontak_awal'] = df_merged.get('kontakA_date')
    df_target['tanggal_wawancara'] = df_merged.get('wawancara_date')
    df_target['tanggal_pembayaran'] = df_merged.get('tanggal_pembayaran')
    df_target['tanggal_masuk'] = df_merged.get('tglmasuk_date')
    df_target['tanggal_keluar'] = df_merged.get('tglkeluar_date')
    
    # 4. ISI KOLOM KOSONG & URUTKAN
    target_columns = [
        'id_calon_jadwal', 'id_calon_akademik', 'tanggal_kontak_awal', 
        'tanggal_wawancara', 'tanggal_pembayaran', 'tanggal_masuk', 'tanggal_keluar'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa_jadwal' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
# Panggil fungsi menggunakan table_3_left_join, calon_siswa, dan calon_siswa_akademik
df_future['calon_siswa_jadwal'] = proses_pindah_ke_calon_siswa_jadwal(
    table_3_left_join, 
    df_future['calon_siswa'], 
    df_future['calon_siswa_akademik']
)

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA_JADWAL MASTER ---")
display(df_future['calon_siswa_jadwal'].head())

🚀 Memulai migrasi data ke: calon_siswa_jadwal...
✅ Migrasi ke 'calon_siswa_jadwal' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA_JADWAL MASTER ---


,id_calon_jadwal,id_calon_akademik,tanggal_kontak_awal,tanggal_wawancara,tanggal_pembayaran,tanggal_masuk,tanggal_keluar
0,1,1,2025-09-15,None,None,2025-09-24,None
1,2,2,2025-09-16,2025-09-16,None,2025-09-25,None
2,3,3,2025-09-16,None,None,None,None
3,4,4,2025-09-17,2025-09-17,None,2025-09-25,None
4,5,5,2025-09-15,2025-09-18,None,None,None


# CALON_SISWA_PROSES

In [45]:
import pandas as pd

def proses_pindah_ke_calon_siswa_proses(df_source, df_calon, df_akademik):
    print("🚀 Memulai migrasi data ke: calon_siswa_proses...")
    
    # 1. PERSIAPAN JEMBATAN ID (Mencari id_calon_akademik)
    df_calon_subset = df_calon[['id_calon', 'nama_lengkap']].copy()
    df_calon_subset['nama_clean'] = df_calon_subset['nama_lengkap'].astype(str).str.lower().str.strip()
    df_calon_subset = df_calon_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    df_akademik_subset = df_akademik[['id_calon_akademik', 'id_calon']].copy()
    df_akademik_subset = df_akademik_subset.drop_duplicates(subset=['id_calon'], keep='first')
    
    # 2. PROSES JOIN BERANTAI
    df_merged = df_source.copy()
    df_merged['nama_clean'] = df_merged['name'].astype(str).str.lower().str.strip()
    
    # Join 1 & 2
    df_merged = df_merged.merge(df_calon_subset[['nama_clean', 'id_calon']], on='nama_clean', how='left')
    df_merged = df_merged.merge(df_akademik_subset, on='id_calon', how='left')
    
    # 3. MAPPING KOLOM PROSES & OPERASIONAL
    df_target = pd.DataFrame()
    df_target['id_calon_siswa_proses'] = range(1, len(df_merged) + 1) # ID Auto-increment
    df_target['id_calon_akademik'] = df_merged['id_calon_akademik']
    
    # Data Admin & PIC
    df_target['admin_pengontak'] = df_merged.get('pengontak_admin')
    df_target['penanggung_jawab'] = df_merged.get('pic')
    
    # Trial & Waktu
    df_target['jenis_trial'] = df_merged.get('trial')
    df_target['hasil_trial'] = df_merged.get('hasil_test')
    df_target['waktu_trial_1'] = df_merged.get('waktu_test1')
    df_target['waktu_trial_2'] = df_merged.get('waktu_test2')
    df_target['tanggal_trial'] = df_merged.get('tanggal_trial')
    df_target['laporan_trial'] = df_merged.get('laporan_trial')
    df_target['placement_trial'] = df_merged.get('placement')
    df_target['lokasi_trial'] = df_merged.get('trial_dimana')
    
    # Status & Pipeline
    df_target['sumber_lead'] = None # Atau sesuaikan jika ada field sumber lead spesifik
    df_target['status_pipeline'] = None # Belum ada data status pipeline, jadi None
    df_target['status_updated_at'] = None # Belum ada data update status, jadi None
    df_target['status_diterima'] = df_merged.get('diterima')
    df_target['status_form_pendaftaran'] = df_merged.get('form_daftar')
    df_target['hasil_penempatan'] = df_merged.get('diterima_dikelas')
    
    # Follow Up & Grup
    df_target['followup_1'] = df_merged.get('followUp1')
    df_target['followup_2'] = df_merged.get('followUp2')
    df_target['followup_3'] = df_merged.get('followUp3')
    df_target['akun_leapverse'] = df_merged.get('akun_lv')
    df_target['wa_grup_leapverse'] = df_merged.get('wag_lv')
    
    # Catatan
    df_target['catatan_admin'] = df_merged.get('catatanadmin')
    df_target['catatan_penting'] = df_merged.get('catatan_penting')
    df_target['keterangan_tambahan'] = df_merged.get('keterangan_til4') # Bisa disesuaikan dengan keterangan_form
    df_target['detail_lainnya'] = df_merged.get('otherDetail')
    
    # 4. ISI KOLOM KOSONG & URUTKAN (Total 27 Kolom)
    target_columns = [
        'id_calon_siswa_proses', 'id_calon_akademik', 'admin_pengontak', 'penanggung_jawab', 
        'jenis_trial', 'hasil_trial', 'waktu_trial_1', 'waktu_trial_2', 'tanggal_trial', 
        'laporan_trial', 'placement_trial', 'lokasi_trial', 'sumber_lead', 'status_pipeline', 
        'status_updated_at', 'status_diterima', 'status_form_pendaftaran', 'hasil_penempatan', 
        'followup_1', 'followup_2', 'followup_3', 'akun_leapverse', 'wa_grup_leapverse', 
        'catatan_admin', 'catatan_penting', 'keterangan_tambahan', 'detail_lainnya'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa_proses' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
df_future['calon_siswa_proses'] = proses_pindah_ke_calon_siswa_proses(
    table_3_left_join, 
    df_future['calon_siswa'], 
    df_future['calon_siswa_akademik']
)

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA_PROSES MASTER ---")
display(df_future['calon_siswa_proses'].head())

🚀 Memulai migrasi data ke: calon_siswa_proses...
✅ Migrasi ke 'calon_siswa_proses' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA_PROSES MASTER ---


,id_calon_siswa_proses,id_calon_akademik,admin_pengontak,penanggung_jawab,jenis_trial,hasil_trial,waktu_trial_1,waktu_trial_2,tanggal_trial,laporan_trial,...,hasil_penempatan,followup_1,followup_2,followup_3,akun_leapverse,wa_grup_leapverse,catatan_admin,catatan_penting,keterangan_tambahan,detail_lainnya
0,1,1,Ibu Sari,None,None,None,0 days 00:00:00,0 days 00:00:00,None,None,...,21 BALLOONS SR1 (QORIN),None,None,None,None,None,None,None,None,None
1,2,2,Bu Dwi,None,None,None,0 days 15:45:00,0 days 00:00:00,None,None,...,None,None,None,None,None,None,None,None,None,None
2,3,3,Ibu Fitri,None,None,None,0 days 00:00:00,0 days 00:00:00,None,None,...,None,None,None,None,None,None,None,None,None,None
3,4,4,Bu Lita,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,5,5,Ibu Fadhil,None,None,None,0 days 00:00:00,0 days 00:00:00,None,None,...,None,None,None,None,None,None,None,None,None,None


# calon_siswa_kursus

In [48]:
import pandas as pd

def proses_pindah_ke_calon_siswa_kursus(df_source, df_calon):
    print("🚀 Memulai migrasi data ke: calon_siswa_kursus...")
    
    # 1. PERSIAPAN JEMBATAN ID (Mencari id_calon)
    df_calon_subset = df_calon[['id_calon', 'nama_lengkap']].copy()
    df_calon_subset['nama_clean'] = df_calon_subset['nama_lengkap'].astype(str).str.lower().str.strip()
    df_calon_subset = df_calon_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    # 2. PROSES JOIN
    df_merged = df_source.copy()
    df_merged['nama_clean'] = df_merged['name'].astype(str).str.lower().str.strip()
    
    # Join untuk mendapatkan id_calon
    df_merged = df_merged.merge(df_calon_subset[['nama_clean', 'id_calon']], on='nama_clean', how='left')
    
    # 3. MAPPING KOLOM
    df_target = pd.DataFrame()
    df_target['id_calon_kursus'] = range(1, len(df_merged) + 1) # ID Auto-increment
    df_target['id_calon'] = df_merged['id_calon']
    
    # Mapping Data Program/Kursus
    df_target['nama_kursus'] = df_merged.get('pilihan_program')
    df_target['jenis_program'] = df_merged.get('jenis_program')
    
    # 4. ISI KOLOM KOSONG & URUTKAN (Total 5 Kolom)
    target_columns = [
        'id_calon_kursus', 'id_calon', 'urutan', 'nama_kursus', 'jenis_program'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa_kursus' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
df_future['calon_siswa_kursus'] = proses_pindah_ke_calon_siswa_kursus(
    table_3_left_join, 
    df_future['calon_siswa']
)

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA_KURSUS MASTER ---")
display(df_future['calon_siswa_kursus'])

🚀 Memulai migrasi data ke: calon_siswa_kursus...
✅ Migrasi ke 'calon_siswa_kursus' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA_KURSUS MASTER ---


,id_calon_kursus,id_calon,urutan,nama_kursus,jenis_program
0,1,1,None,English,GE
1,2,2,None,English,GE
2,3,3,None,Digital,COD
3,4,4,None,English,GE
4,5,5,None,English,GE
...,...,...,...,...,...
161,162,162,None,None,None
162,163,163,None,None,None
163,164,164,None,None,None
164,165,165,None,None,None


# calon_siswa_bayar

In [50]:
import pandas as pd

def proses_pindah_ke_calon_siswa_bayar(df_source, df_calon, df_akademik):
    print("🚀 Memulai migrasi data ke: calon_siswa_bayar...")
    
    # 1. PERSIAPAN JEMBATAN ID (Mencari id_calon_akademik)
    df_calon_subset = df_calon[['id_calon', 'nama_lengkap']].copy()
    df_calon_subset['nama_clean'] = df_calon_subset['nama_lengkap'].astype(str).str.lower().str.strip()
    df_calon_subset = df_calon_subset.drop_duplicates(subset=['nama_clean'], keep='first')
    
    df_akademik_subset = df_akademik[['id_calon_akademik', 'id_calon']].copy()
    df_akademik_subset = df_akademik_subset.drop_duplicates(subset=['id_calon'], keep='first')
    
    # 2. PROSES JOIN BERANTAI
    df_merged = df_source.copy()
    df_merged['nama_clean'] = df_merged['name'].astype(str).str.lower().str.strip()
    
    # Join 1 & 2
    df_merged = df_merged.merge(df_calon_subset[['nama_clean', 'id_calon']], on='nama_clean', how='left')
    df_merged = df_merged.merge(df_akademik_subset, on='id_calon', how='left')
    
    # 3. MAPPING KOLOM ADMINISTRASI KEUANGAN
    df_target = pd.DataFrame()
    df_target['id_calon_bayar'] = range(1, len(df_merged) + 1) # ID Auto-increment
    df_target['id_calon_akademik'] = df_merged['id_calon_akademik']
    
    # Pemetaan kolom data keuangan dari table_3_left_join
    df_target['nomor_invoice'] = df_merged.get('nomor_invoice')
    df_target['bank_pembayaran'] = df_merged.get('bank')
    df_target['tanggal_konfirmasi_bayar'] = df_merged.get('tanggal_pembayaran')
    df_target['bulan_mulai_belajar'] = df_merged.get('bulan_masuk')
    df_target['lokasi_belajar'] = df_merged.get('lokasi')
    
    # 4. ISI KOLOM KOSONG & URUTKAN (Memastikan total 7 Kolom)
    target_columns = [
        'id_calon_bayar', 'id_calon_akademik', 'nomor_invoice', 
        'bank_pembayaran', 'tanggal_konfirmasi_bayar', 
        'bulan_mulai_belajar', 'lokasi_belajar'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'calon_siswa_bayar' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
df_future['calon_siswa_bayar'] = proses_pindah_ke_calon_siswa_bayar(
    table_3_left_join, 
    df_future['calon_siswa'], 
    df_future['calon_siswa_akademik']
)

# Audit Akhir
print("\n--- PRATINJAU CALON_SISWA_BAYAR MASTER ---")
display(df_future['calon_siswa_bayar'].head())

🚀 Memulai migrasi data ke: calon_siswa_bayar...
✅ Migrasi ke 'calon_siswa_bayar' selesai! 166 data berhasil diproses.

--- PRATINJAU CALON_SISWA_BAYAR MASTER ---


,id_calon_bayar,id_calon_akademik,nomor_invoice,bank_pembayaran,tanggal_konfirmasi_bayar,bulan_mulai_belajar,lokasi_belajar
0,1,1,None,Mandiri,None,September,Sby
1,2,2,None,Mandiri,None,September,Sby
2,3,3,None,None,None,None,None
3,4,4,None,None,None,None,None
4,5,5,None,Mandiri,None,September,Sby


# peminjaman

In [53]:
import pandas as pd

def proses_pindah_ke_peminjaman(df_source):
    print("🚀 Memulai migrasi data ke: peminjaman...")
    
    # 1. BUAT DATAFRAME TARGET & MAPPING KOLOM
    df_target = pd.DataFrame()
    
    # Pemetaan 1-to-1 dari df_old['pinjam'] ke df_future['peminjaman']
    df_target['id_pinjam'] = df_source['idpinjam']
    df_target['tanggal_pinjam'] = df_source['tglpinjam']
    df_target['keperluan'] = df_source['deskripsi']
    df_target['id_user'] = df_source['idusers']
    df_target['status_pinjam'] = df_source['status']
    df_target['catatan_sarpras'] = df_source['catatan']
    df_target['created_at'] = df_source['created_at']
    
    # 2. PASTIKAN STRUKTUR & URUTAN KOLOM TEPAT (7 Kolom)
    target_columns = [
        'id_pinjam', 'tanggal_pinjam', 'keperluan', 'id_user', 
        'status_pinjam', 'catatan_sarpras', 'created_at'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    # Terapkan urutan kolom
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'peminjaman' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
# Panggil fungsi menggunakan df_old['pinjam']
df_future['peminjaman'] = proses_pindah_ke_peminjaman(df_old['pinjam'])

# Audit Akhir
print("\n--- PRATINJAU PEMINJAMAN MASTER ---")
display(df_future['peminjaman'].head())

🚀 Memulai migrasi data ke: peminjaman...
✅ Migrasi ke 'peminjaman' selesai! 194 data berhasil diproses.

--- PRATINJAU PEMINJAMAN MASTER ---


,id_pinjam,tanggal_pinjam,keperluan,id_user,status_pinjam,catatan_sarpras,created_at
0,2,2023-06-11,<p>pinjam kamera - fun class tk mitra - 1 - 11...,U00026,Selesai,,2023-06-12 13:20:39
1,3,2023-06-11,<p>1. kamera - fun class TK mitra - 1 - 11 Jun...,U00026,Selesai,,2023-06-12 13:21:35
2,5,2023-08-21,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Mid Te...,U00026,Selesai,,2023-08-16 15:01:55
3,6,2023-08-23,<p>Pinjam kamera untuk rekaman video checklist...,U00033,Selesai,,2023-08-23 10:19:12
4,7,2023-10-11,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Final ...,U00026,Selesai,,2023-10-10 15:30:53


# pengadaan

In [58]:
import pandas as pd

def proses_pindah_ke_pengadaan(df_source):
    print("🚀 Memulai migrasi data ke: pengadaan...")
    
    # 1. BUAT DATAFRAME TARGET & MAPPING KOLOM 1-TO-1
    df_target = pd.DataFrame()
    
    df_target['id_pengadaan'] = df_source['idbeli']
    df_target['deskripsi'] = df_source['deskripsi']
    df_target['url_produk'] = df_source['link']
    df_target['id_user'] = df_source['idusers']
    df_target['status_pengajuan'] = df_source['status']
    df_target['catatan_admin'] = df_source['catatan']
    df_target['tanggal_pengajuan'] = df_source['created_at']
    df_target['tanggal_selesai'] = df_source['done_at']
    df_target['url_pembelian'] = df_source['linkpurchase']
    
    # 2. PASTIKAN STRUKTUR & URUTAN KOLOM TEPAT (9 Kolom)
    target_columns = [
        'id_pengadaan', 'deskripsi', 'url_produk', 'id_user', 
        'status_pengajuan', 'catatan_admin', 'tanggal_pengajuan', 
        'tanggal_selesai', 'url_pembelian'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    # Terapkan urutan kolom sesuai skema database masa depan
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'pengadaan' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
# Panggil fungsi menggunakan df_old['purchase']
df_future['pengadaan'] = proses_pindah_ke_pengadaan(df_old['purchase'])

# Audit Akhir
print("\n--- PRATINJAU PENGADAAN MASTER ---")
display(df_future['pengadaan'].head())

🚀 Memulai migrasi data ke: pengadaan...
✅ Migrasi ke 'pengadaan' selesai! 110 data berhasil diproses.

--- PRATINJAU PENGADAAN MASTER ---


,id_pengadaan,deskripsi,url_produk,id_user,status_pengajuan,catatan_admin,tanggal_pengajuan,tanggal_selesai,url_pembelian
0,18,"<p><span style=""font-family: Arial; font-size:...",https://docs.google.com/spreadsheets/d/1B0sVeV...,U00012,Selesai,,2023-07-06 11:14:15,NaT,https://docs.google.com/spreadsheets/d/15Xuh2Z...
1,20,"<p>""KABEL TELEPON</p>\r\n<p>kabel roset telepo...",https://docs.google.com/spreadsheets/d/1B0sVeV...,U00012,Selesai,,2023-08-03 09:36:19,NaT,https://docs.google.com/spreadsheets/d/15Xuh2Z...
2,21,<p>15 pcs Sarung kursi untuk Lab Komputer (10 ...,https://docs.google.com/spreadsheets/d/1Onr3rr...,U00001,Selesai,,2023-08-22 09:58:34,NaT,https://docs.google.com/spreadsheets/d/15Xuh2Z...
3,22,<p>Pembelian 48 pcs Landyard</p>,https://docs.google.com/spreadsheets/d/1Onr3rr...,U00001,Selesai,,2023-08-22 10:42:37,NaT,https://docs.google.com/spreadsheets/d/15Xuh2Z...
4,23,"<p>1 ""HEADPHONE JACK</p>\n<p>MBOISGET - PREMIU...",https://docs.google.com/spreadsheets/d/1B0sVeV...,U00012,Selesai,,2023-08-28 16:03:09,2023-09-10,https://docs.google.com/spreadsheets/d/15Xuh2Z...


# problem

In [59]:
import pandas as pd

def proses_pindah_ke_problem(df_source):
    print("🚀 Memulai migrasi data ke: problem...")
    
    # 1. BUAT DATAFRAME TARGET & MAPPING KOLOM 1-TO-1
    df_target = pd.DataFrame()
    
    df_target['id_problem'] = df_source['idproblem']
    df_target['detail_masalah'] = df_source['keterangan']
    df_target['id_user'] = df_source['idusers']
    df_target['status_perbaikan'] = df_source['status']
    df_target['tanggal_lapor'] = df_source['created_at']
    df_target['tanggal_selesai'] = df_source['solved_at']
    df_target['catatan_teknisi'] = df_source['catatan']
    df_target['gambar_problem'] = df_source['image_path']
    
    # 2. PASTIKAN STRUKTUR & URUTAN KOLOM TEPAT (8 Kolom)
    target_columns = [
        'id_problem', 'detail_masalah', 'id_user', 'status_perbaikan', 
        'tanggal_lapor', 'tanggal_selesai', 'catatan_teknisi', 'gambar_problem'
    ]
    
    for col in target_columns:
        if col not in df_target.columns:
            df_target[col] = None
            
    # Terapkan urutan kolom sesuai skema database masa depan
    df_target = df_target[target_columns]
    
    print(f"✅ Migrasi ke 'problem' selesai! {len(df_target)} data berhasil diproses.")
    return df_target

# === EKSEKUSI ===
# Panggil fungsi menggunakan df_old['problem']
df_future['problem'] = proses_pindah_ke_problem(df_old['problem'])

# Audit Akhir
print("\n--- PRATINJAU PROBLEM MASTER ---")
display(df_future['problem'].head())

🚀 Memulai migrasi data ke: problem...
✅ Migrasi ke 'problem' selesai! 160 data berhasil diproses.

--- PRATINJAU PROBLEM MASTER ---


,id_problem,detail_masalah,id_user,status_perbaikan,tanggal_lapor,tanggal_selesai,catatan_teknisi,gambar_problem
0,43,AC Kelas Miss Erika kurang dingin ( belakang,U00012,Proses,2023-06-28 16:07:52,NaT,"sudah info ke Pak Irawan,\r\nTukang AC masih l...",None
1,58,Boya/mic untuk kelas hybrid tidak berfungsi (,U00026,Terselesaikan,2023-07-06 10:45:03,2023-07-17,,None
2,60,tegangan listrik di ruang kelas belakang dapur...,U00033,Terselesaikan,2023-07-13 16:59:57,2023-08-10,pemberian stabilizer,None
3,61,ac brisik,U00033,Terselesaikan,2023-07-14 09:24:29,2023-07-25,sudah tidak berisik\r\n,None
4,62,Kabel power monitor PC room 2 longgar. Saat me...,U00036,Terselesaikan,2023-07-17 15:04:01,2023-07-17,,None


# Pickle

In [60]:
import json
import pickle
with open('fase_3_cimut.pkl', 'wb') as f:
    pickle.dump(df_new, f)

print("✓ Data df_new sudah disimpan ke df_new.pkl")
print("Siap untuk digunakan di insert_handler.ipynb")

✓ Data df_new sudah disimpan ke df_new.pkl
Siap untuk digunakan di insert_handler.ipynb
